<a href="https://colab.research.google.com/github/klausreitz/cohid/blob/main/Vaz%C3%B5es_Naturais.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



#**Automação de tarefas do processo de avaliação de proposições de séries de vazões naturais médias mensais de empreendimentos do setor elétrico**
##Estudo de caso aplicado: UHE Foz do Areia.


###**SEÇÃO 0: Configurações Preliminares de Ambiente e Infraestrutura**
####Nesta seção, é estabelecido o alicerce tecnológico do projeto. Sua execução é obrigatória e deve ser a primeira do fluxo, pois prepara o runtime do Google Colab para as operações de Inteligência Artificial e processamento hídrico que ocorrerão nas etapas seguintes.

###**SEÇÃO 0.0**: Configurações de Runtime e Logging
####Estabelece o nível de verbosidade do sistema (INFO) para monitoramento em tempo real e aplica correções de assincronicidade via nest_asyncio, permitindo que o pipeline RAG execute loops sem interrupções no kernel.

###**SEÇÃO 0.1**: Instalação de Dependências de Sistema e Python
####Instalação dos motores de baixo nível para visão computacional e manipulação de documentos:

####Sistêmicos: Tesseract OCR (reconhecimento de caracteres) e Poppler/Ghostscript (renderização de PDF).

####Bibliotecas Python: Frameworks de IA (LlamaIndex), busca vetorial (FAISS), conectores de modelos (Groq, HuggingFace) e utilitários de documentos (PyMuPDF, python-docx).

###**SEÇÃO 0.2**: Definição de Funções de Uso Comum
####Implementação da lógica de apoio que será utilizada em todo o pipeline, incluindo:

####Algoritmos de localização dinâmica de tabelas de vazões em PDFs.

####Manipuladores de persistência em Excel e validação de estruturas de diretórios.

####Verificadores de integridade para chaves de API e dimensões de vetorização (embeddings).

###**SEÇÃO 0.3**: Mapeamento de Caminhos e Estruturação do Objeto de Contexto
####Montagem do Google Drive: Vinculação do sistema de arquivos para persistência.

####Definição do PIPELINE_CONTEXT: Criação do "Dicionário Mestre" que centraliza todas as variáveis, caminhos de rede, parâmetros de modelos (E5-Large e Llama-3.3-70B) e o mapa de extração de dados para a Nota Técnica.

###**SEÇÃO 0.4**: Validação e Segurança
####Execução de protocolos de checagem para garantir que todas as pastas, arquivos e chaves de API estão operacionais, prevenindo falhas de execução no meio do processamento pesado.

In [ ]:
# SEÇÃO 0: CONFIGURAÇÕES PRELIMINARES DE AMBIENTE, INSTALAÇÕES,MONTAGEM DO DRIVE E MAPEAMENTO DE ARQUIVOS

#Subseção 0.0 Configurações preliminares de ambiente (LOGGING nível INFO)
print("💡 Configurações preliminares de ambiente...")
import logging
import sys
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))
print("✅ Logging configurado globalmente para nível INFO.\n\n💡 Iniciando instalação de bibliotecas...")

# Subseção 0.1 Instalação de dependências de sistema e bibliotecas Python

!sudo apt update
!sudo apt install -y -q poppler-utils tesseract-ocr tesseract-ocr-por
!apt-get install -y -q ghostscript

!pip install -q \
    camelot-py \
    faiss-cpu \
    "huggingface-hub<1.0.1" \
    jedi \
    llama-cloud-services \
    llama-index \
    llama-index-core \
    llama-index-embeddings-huggingface \
    llama-index-llms-groq \
    llama-index-readers-pdf-table \
    llama-index-vector-stores-faiss \
    llama-parse \
    nest_asyncio \
    openpyxl \
    pandas \
    pdf2image \
    pdfplumber \
    Pillow \
    pymupdf \
    pypdf \
    pytesseract \
    python-docx

print("✅ Bibliotecas instaladas.\n\n💡 Iniciando importação de bibliotecas e preparação do runtime...")

# Subseção 0.2 Importação de bibliotecas e preparação do runtime

import nest_asyncio
nest_asyncio.apply() # prevenção a problemas de assincronicidade, aninha loops do pipeline RAG com o event loop do Kernel

from collections import Counter
from dataclasses import dataclass
from datetime import datetime
from docx import Document
from google.colab import drive
from google.colab import userdata
from io import StringIO
from llama_index.core import (
    Document,
    load_index_from_storage,
    Response,
    Settings,
    SimpleDirectoryReader,
    StorageContext,
    VectorStoreIndex
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.prompts import PromptTemplate
from llama_index.core.workflow import Workflow
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq
from llama_index.readers.pdf_table import PDFTableReader
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_parse import LlamaParse
from openpyxl import load_workbook
from openpyxl.formatting.rule import FormulaRule
from openpyxl.styles import Alignment, PatternFill, Font
from pathlib import Path
from pdf2image import convert_from_path
from pypdf import PdfReader
from tqdm.auto import tqdm

import asyncio
import docx
import faiss
import fitz
import json
import numpy as np
import openpyxl
import os
import pandas as pd
import pdfplumber
import pickle
import pprint
import pytesseract
import re
import shutil
import sys
import tarfile
import textwrap
import time

#Subseção 0.3 Montagem do Drive e estrutura de pastas do projeto

print("✅ Bibliotecas importadas e nest_asyncio aplicado...\n\n💡 Iniciando montagem do Drive e estrutura de pastas do projeto...")
drive.mount('/content/drive')
RAIZ = '/content/drive/MyDrive/colab_notebooks/automacao_nota_tecnica'
ARQUIVO_OUTORGA = os.path.join(RAIZ, 'outorga.pdf')
ARQUIVO_PDF_ONS = os.path.join(RAIZ, 'Atualização de Séries Históricas de Vazões - 1931 a 2024.pdf')
ARQUIVO_EXCEL_ONS = os.path.join(RAIZ, 'Vazões_Mensais_1931_2024.xls')
ARQUIVO_PROCESSO = os.path.join(RAIZ, 'processo.pdf')
COMPARA = os.path.join(RAIZ, "COMPARA.xlsx")
LOCAL_PERSIST_DIR = "/content/faiss_index_storage"
NOME_BACKUP = "faiss_index_storage.tar.gz"
ARQUIVO_COMPACTADO_LOCAL = os.path.join("/content", NOME_BACKUP)
ARQUIVO_COMPACTADO_DRIVE = os.path.join(RAIZ, NOME_BACKUP)
ARQUIVO_PARSED_PKL = os.path.join(RAIZ, "docs_parsed.pkl")
TEMPLATE = os.path.join(RAIZ, "NOTA_TECNICA_UHE_Template.docx")
print(f"Caminho Completo do Processo: {ARQUIVO_PROCESSO}")
print(f"Caminho Completo da Outorga: {ARQUIVO_OUTORGA}")
print(f"✅ Google Drive montado. Projeto alocado em: {RAIZ} \n\n💡 Iniciando definição de funções de uso comum...")

# Função para procurar a tabela de dados do empreendedor no PDF

def localizar_paginas_vazoes_dinamico(caminho_pdf):
    if not os.path.exists(caminho_pdf):
        print(f"❌ Erro: Arquivo não encontrado em {caminho_pdf}")
        return None

    leitor = PdfReader(caminho_pdf)
    total_paginas = len(leitor.pages)
    reader = PDFTableReader()
    paginas_detectadas = []

    for p in tqdm(range(1, total_paginas + 1), desc="Buscando Tabelas"):
        try:
            docs = reader.load_data(file=Path(caminho_pdf), pages=str(p))
            for doc in docs:
                linhas = doc.text.strip().split("\n")
                if len(linhas) < 3: continue # Reduzi o mínimo para não ignorar páginas curtas

                # Extrai apenas números da primeira "coluna" de cada linha
                anos_na_pagina = []
                for linha in linhas:
                    # Pega o primeiro elemento antes da vírgula ou espaço
                    primeira_parte = re.split(r'[,\s\t]', linha.strip())[0]
                    # Limpa tudo que não for número
                    apenas_numeros = re.sub(r'\D', '', primeira_parte)

                    if apenas_numeros and 1930 <= int(apenas_numeros) <= 2026:
                        anos_na_pagina.append(int(apenas_numeros))

                # Se houver pelo menos 3 anos sequenciais ou válidos, a página é útil
                if len(anos_na_pagina) >= 3:
                    paginas_detectadas.append(p)
                    break
        except:
            continue
    PIPELINE_CONTEXT["extracao_processo"]["pagina_carta"] = paginas_detectadas[0] - 2 if paginas_detectadas else None
    if not paginas_detectadas: return None
    return ",".join(map(str, sorted(list(set(paginas_detectadas)))))

# Função para salvar dados de dataframes em planilha Excel

def salvar_aba_excel(df, caminho_excel, nome_aba):
    """
    Salva um DataFrame em uma aba específica de um Excel.
    - Cria o arquivo se não existir
    - Substitui apenas a aba informada
    - Preserva todas as demais abas

    Exemplo de uso:
        salvar_aba_excel(
            df=df_final,
            caminho_excel=PIPELINE_CONTEXT["COMPARA"],
            nome_aba='APROVEITAMENTOS'
        )
    """

    if os.path.exists(caminho_excel):
        modo = "a"
        if_sheet = "replace"
    else:
        modo = "w"
        if_sheet = None

    with pd.ExcelWriter(
        caminho_excel,
        engine="openpyxl",
        mode=modo,
        if_sheet_exists=if_sheet
    ) as writer:
        df.to_excel(writer, sheet_name=nome_aba, index=False)

    print(f"✅ Aba '{nome_aba}' salva/atualizada com sucesso.")
    return True

# Função de teste da planilha Excel:

def diagnosticar_excel(caminho_excel, nome_aba):
    """
    Verifica se um arquivo Excel existe e se uma aba específica está presente.

    Parâmetros:
    - caminho_excel (str): caminho completo do arquivo Excel
    - nome_aba (str): nome da aba a ser verificada

    Exemplo de uso:
        diagnosticar_excel(
            caminho_excel=CAMINHO_EXCEL,
            nome_aba='APROVEITAMENTOS'
        )
    """
    if not os.path.exists(caminho_excel):
        print("❌ Arquivo Excel não encontrado.")
        return

    xls = pd.ExcelFile(caminho_excel)
    print(f"🧾 Abas existentes: {xls.sheet_names}")

    if nome_aba in xls.sheet_names:
        print(f"✅ Aba '{nome_aba}' confirmada.")
    else:
        print(f"❌ Aba '{nome_aba}' não encontrada.")

#Função para validação da estrutura do projeto no Drive

def validar_estrutura_projeto(caminho_raiz: str, subdirs: dict) -> None:
    print("📂 Validando/criando estrutura de diretórios do projeto...")
    print(f"    Caminho raiz: {caminho_raiz}")

    # 1. Validação de tipo: agora esperamos um dicionário
    if not isinstance(subdirs, dict):
        raise TypeError(f"'subdirs' deve ser um dicionário, mas recebeu {type(subdirs).__name__}")

    # 2. Percorrer as chaves do dicionário
    for chave, valor in subdirs.items():
        nome_pasta = valor if isinstance(valor, str) else chave
        caminho_completo = os.path.join(caminho_raiz, nome_pasta)
        print(f"    - Garantindo subdiretório: {nome_pasta}")
        os.makedirs(caminho_completo, exist_ok=True)

    print("✅ Estrutura de diretórios validada.\n")

def validar_chaves_api(userdata):
    try:
        GROQ_API_KEY = userdata.get("GROQ_API_KEY")
        if not GROQ_API_KEY:
            raise RuntimeError("GROQ_API_KEY não encontrada em secrets")

        os.environ["GROQ_API_KEY"] = GROQ_API_KEY
        print("   - GROQ_API_KEY carregada")

        LlamaCloud_API_KEY = userdata.get("LlamaCloud_API_KEY")
        if not LlamaCloud_API_KEY:
            raise RuntimeError("LlamaCloud_API_KEY não encontrada em secrets")

        os.environ["LLAMA_CLOUD_API_KEY"] = LlamaCloud_API_KEY
        print("   - LlamaCloud_API_KEY carregada")

        HF_TOKEN = userdata.get("HF_TOKEN")
        if not HF_TOKEN:
            raise RuntimeError("HF_TOKEN não encontrada em secrets")

        os.environ["HF_TOKEN"] = HF_TOKEN
        print("   - HF_TOKEN carregada")

        print("✅ Todas as chaves API foram validadas e carregadas com sucesso.\n")

    except Exception as e:
        print("❌ ERRO CRÍTICO na validação das chaves de API")
        raise

# Função de validação de dimensões de Embedding

def validar_dimensao_embedding(embed_cfg):
    expected_dim = embed_cfg.get("embed_dim")

    if not expected_dim:
        print("ℹ️ Dimensão esperada não definida no contexto. Validação ignorada.")
        return

    try:
        real_dim = Settings.embed_model._model.get_sentence_embedding_dimension()
    except Exception as e:
        raise RuntimeError(
            f"Não foi possível obter a dimensão real do embedding: {e}"
        )

    if real_dim != expected_dim:
        raise RuntimeError(
            f"Dimensão do embedding divergente: esperado={expected_dim}, real={real_dim}"
        )

    print(f"✅ Dimensões da vetorização (Embedding) validadas ({real_dim})")

# Função de validação de contexto

def validar_contexto_completo(ctx: dict) -> None:
    erros = []
    raiz_atual = ctx.get("execucao", {}).get("paths", {}).get("raiz_projeto")

    # Verificação de segurança: o caminho no contexto bate com a sua variável global?
    if raiz_atual != RAIZ:
        print(f"⚠️ Aviso: O caminho no contexto difere da variável global RAIZ.")
    if not raiz_atual or not os.path.isdir(raiz_atual):
        raise RuntimeError(f"❌ Erro Crítico: Raiz do projeto inválida: {raiz_atual}")

    # 1. Pegamos a raiz (fundamental para o resto)
    paths_dict = ctx.get("execucao", {}).get("paths", {})
    raiz = paths_dict.get("raiz_projeto")

    if not raiz or not os.path.isdir(raiz):
        raise RuntimeError(f"❌ Erro Crítico: Raiz do projeto inválida ou inacessível: {raiz}")

    print("💡 Verificando pilares obrigatórios e caminhos...")

    # --- A. VALIDAÇÃO DE CAMINHOS (Obrigatórios vs Opcionais) ---
    obrigatorios = ["raiz_projeto", "pdf_processo", "pdf_outorga"]

    for chave, valor in paths_dict.items():
        # Se for obrigatório e estiver vazio
        if chave in obrigatorios and not valor:
            erros.append(f"Configuração '{chave}' é obrigatória e está vazia.")
            continue

    # --- C. VALIDAÇÃO DE LÓGICA E ESTRUTURA ---
    print("💡 Analisando lógica de OCR e respostas...")

    # Validação OCR
    docs = ctx.get("documentos", {})
    for nome, cfg in docs.items():
        if cfg.get("ocr_habilitado") and not cfg.get("arquivo_original"):
            erros.append(f"Documento '{nome}' exige OCR mas está sem arquivo original.")

    # Validação Perguntas
    if not isinstance(ctx.get("query_engine", {}).get("perguntas_padrao"), list):
        erros.append("O campo 'perguntas_padrao' deve ser uma lista.")

    # Validação de Chaves [a] a [p]
    respostas = ctx.get("extracao_processo", {}).get("respostas", {})
    for letra in "abcdefghijklmnop":
        chave = f"[{letra}]"
        if chave not in respostas:
            erros.append(f"Mapa de extração incompleto: Falta a chave '{chave}'.")

    # --- D. VEREDITO FINAL ---
    if erros:
        print("\n🛑 FALHAS DE VALIDAÇÃO ENCONTRADAS:")
        for e in erros:
            print(f"   - {e}")
        # Usamos RuntimeError para parar tudo se algo estiver errado
        raise RuntimeError("Contexto do pipeline reprovado na validação.")

    print("✅ Contexto validado com sucesso. Tudo pronto para o processamento!")

# Função de aplicação de perfil de execução em relação à verbosidade, debug e rastreabilidade

def aplicar_exec_profile(profile):
    print("🔧 Aplicando perfil de execução...")
    print(f"   verbose={profile.verbose}, debug={profile.debug}, "
          f"show_progress={profile.show_progress}, show_source={profile.show_source}")
    Settings.verbose = profile.verbose
    Settings.debug = profile.debug
    Settings.show_progress = profile.show_progress
    Settings.show_source = profile.show_source

    print(f"✅ Perfil de execução aplicado: '{profile.name}'.\n")

print("✅ Funções de uso comum definidas.\n\n💡 Definindo perfil de verbosidade, debug e rastreabilidade...")

# Função de extração de série de vazões de PDF

def extrair_vazoes(caminho_pdf, paginas):
    reader = PDFTableReader()
    todos_os_dfs = []
    lista_paginas = str(paginas).split(',')

    colunas_finais = ["Ano", "jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]

    for p in lista_paginas:
        p = p.strip()
        print(f"📖 Processando página {p}...")
        docs = reader.load_data(file=Path(caminho_pdf), pages=p)

        for doc in docs:
            try:
                # Lemos SEM cabeçalho
                df_temp = pd.read_csv(StringIO(doc.text), header=None)
                df_temp = df_temp.dropna(axis=1, how='all').dropna(axis=0, how='all')

                if df_temp.empty: continue

                primeira_celula = str(df_temp.iloc[0, 0])
                valor_limpo = re.sub(r'\D', '', primeira_celula)
                is_ano = valor_limpo.isdigit() and 1930 <= int(valor_limpo) <= 2026

                if not is_ano:
                    df_temp = df_temp.iloc[1:]

                # Garante 13 colunas
                df_temp = df_temp.iloc[:, :13]
                todos_os_dfs.append(df_temp)
            except Exception as e:
                print(f"   ⚠️ Erro na pág {p}: {e}")

    if not todos_os_dfs: return None

    df_final = pd.concat(todos_os_dfs, ignore_index=True)
    df_final.columns = colunas_finais

    # --- NOVA ETAPA: NORMALIZAÇÃO NUMÉRICA PESADA ---
    for col in df_final.columns:
        # 1. Converte para string e limpa espaços nas pontas
        df_final[col] = df_final[col].astype(str).str.strip()

        if col == "Ano":
            # Ano: Mantém apenas dígitos
            df_final[col] = pd.to_numeric(df_final[col].str.replace(r'\D', '', regex=True), errors='coerce')
        else:
            # Meses: Troca vírgula por ponto e remove qualquer caractere que não seja número, ponto ou sinal de menos
            df_final[col] = df_final[col].str.replace(',', '.')
            df_final[col] = df_final[col].str.replace(r'[^0-9.\-]', '', regex=True)
            df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

    # Remove linhas onde o Ano não foi identificado
    df_final = df_final.dropna(subset=['Ano'])
    df_final['Ano'] = df_final['Ano'].astype(int)

    return df_final

# Transferir valores detectados para a planilha 'NOTA' que alimenta a minuta da Nota Técnica

def gravar_valor_detectado(caminho_excel, valor, linha, coluna):
    """
    Grava um valor em uma célula específica da aba 'NOTA'.
    Exemplo: gravar_valor_detectado(COMPARA, "Rio Amazonas", 5, 5) # Grava na E5
    """
    if not os.path.exists(caminho_excel):
        print(f"Erro: O arquivo {caminho_excel} não foi encontrado.")
        return

    # Carrega o workbook
    wb = openpyxl.load_workbook(caminho_excel)

    if PIPELINE_CONTEXT["extracao_nota"]["nome_aba"] not in wb.sheetnames:
        print(f"Erro: A aba '{PIPELINE_CONTEXT["extracao_nota"]["nome_aba"]}' não existe no arquivo.")
        return

    ws_nota = wb[PIPELINE_CONTEXT["extracao_nota"]["nome_aba"]]

    # Grava o valor usando as variáveis 'linha' e 'coluna'
    ws_nota.cell(row=linha, column=coluna).value = valor

    # Salva o arquivo
    wb.save(caminho_excel)

    # O print usa f-string para mostrar a posição exata
    print(f"Sucesso: Valor '{valor}' gravado na Linha {linha}, Coluna {coluna} da aba {PIPELINE_CONTEXT["extracao_nota"]["nome_aba"]}.")

# Perfis de execução que controlam verbosidade, debug e rastreabilidade

@dataclass(frozen=True)
class ExecProfile:
    name: str
    verbose: bool
    debug: bool
    show_progress: bool
    show_source: bool

AUDITORIA = ExecProfile(
    name="AUDITORIA",
    verbose=True,
    debug=True,
    show_progress=True,
    show_source=True
)

OPERACAO = ExecProfile(
    name="OPERAÇÃO",
    verbose=True,
    debug=False,
    show_progress=True,
    show_source=False
)

PRODUCAO = ExecProfile(
    name="PRODUÇÃO",
    verbose=False,
    debug=False,
    show_progress=False,
    show_source=False
)

aplicar_exec_profile(PRODUCAO)

PIPELINE_CONTEXT = {
    "ambiente": {
        "limite_caracteres_legiveis": 50,   # heurística OCR por página
        "chunking": {
            "size": 512,
            "overlap": 20
        },
        "encoding": {
            "primeira_opcao": "utf-8",
            "segunda_opcao": "latin-1",
            "terceira_opcao": "cp1252"
        },
        "indent" : [4
        ]
    },
    "execucao": {
        "paths": {
            "raiz_projeto": RAIZ,
            "excel_saida": COMPARA,
            "excel_ons": ARQUIVO_EXCEL_ONS,
            "pdf_processo": ARQUIVO_PROCESSO,
            "pdf_outorga": ARQUIVO_OUTORGA,
            "pdf_ons": ARQUIVO_PDF_ONS,
            "nota_template": TEMPLATE,
            "persistencia_drive": ARQUIVO_PARSED_PKL,
            "persistencia_local": LOCAL_PERSIST_DIR,
            "compactado_local": ARQUIVO_COMPACTADO_LOCAL,
            "compactado_drive": ARQUIVO_COMPACTADO_DRIVE,
            "diretorio_faiss_drive": os.path.join(RAIZ, "faiss_index_storage"),
            "nota_final": None, # A ser definido caso a caso, automaticamente
            "diretorio_uhe_trabalho": None, # A ser definido caso a caso, automaticamente
            "pdf_processo_ocr": None, # A ser definido caso a caso, automaticamente
            "pdf_outorga_ocr": None # A ser definido caso a caso, automaticamente
        },
    },
    "documentos": {
        "nota_tecnica": {
            "arquivo_original": TEMPLATE,
            "nome_arquivo_final": None, # A ser definido caso a caso, automaticamente
            "tipo": "nao-estruturado",
            "ocr_habilitado": False
        },
        "processo": {
            "arquivo_original": ARQUIVO_PROCESSO,
            "tipo": "nao-estruturado",
            "ocr_habilitado": True
        },
        "processo_ocr": {
            "arquivo_original": ARQUIVO_PROCESSO,
            "tipo": "nao-estruturado",
            "ocr_habilitado": True
        },
        "outorga": {
            "arquivo_original": ARQUIVO_OUTORGA,
            "tipo": "nao-estruturado",
            "ocr_habilitado": True
        },
        "outorga_ocr": {
            "arquivo_original": ARQUIVO_OUTORGA,
            "tipo": "nao-estruturado",
            "ocr_habilitado": True
        },
        "aproveitamentos": {
            "arquivo_original": ARQUIVO_PDF_ONS,
            "tipo": "nao-estruturado",
            "ocr_habilitado": False
        },
        "ons": {
            "arquivo_original": ARQUIVO_EXCEL_ONS,
            "tipo": "estruturado"
        }
    },
    "modelos": {
        "embedding": {
            "provider": HuggingFaceEmbedding,
            "params": {
                "model_name": "intfloat/multilingual-e5-large",
                "normalize": True,
                "embed_batch_size": 32,
                "show_progress_bar": True
            },
            "embed_dim": 1024
        },
        "llm": {
            "provider": Groq,
            "model": "llama-3.3-70b-versatile"
        }
    },
    "subdirs": {
        "UHE": {
            "caso": None
        }
    },
    "query_engine": {
        "perguntas_padrao": [
            "Qual é o nome do empreendimento, da usina hidrelétrica?",
            "Qual é o rio onde está localizado o empreendimento?",
            "Qual é o responsável pelo empreendimento?",
            "Em que COMUNICAÇÃO INTERNA é solicitada a avaliação da série de vazões?",
            "Qual é o Documento nº da COMUNICAÇÃO INTERNA?",
            "O que é solicitado para a SHE fazer com a série de vazões?",
            "Número do processo?",
            "Número do requerimento de outorga?",
            "De quem é a carta com a tabela de vazões?",
            "Qual é a carta cujo anexo tem as vazões?",
            "De quando é a carta?",
            "A carta está em que página?"
        ],
        "system_prompt": """
            Você é um assistente RAG especialista em análise de documentos de processos regulatórios.
            Responda à PERGUNTA utilizando APENAS as informações do CONTEXTO fornecido.
            Não invente, não infira além do texto, não utilize conhecimento externo.

            REGRAS OBRIGATÓRIAS:
            - Caso existam múltiplas respostas para a mesma pergunta, liste todas separadas por ponto e vírgula com a citação de fonte de cada uma.
            - Para cada informação afirmada, cite explicitamente a fonte no formato:
              (Fonte: [file_name], Pág. [page_number])
            - Se a informação não estiver no contexto, responda:
              "Não localizei a resposta."
        """,
        "similarity_top_k": 5,
        "nome_aba": "RAG_respostas",
        "persistencia": COMPARA
    },
    "extracao_processo": {
        "total_paginas": None,
        "paginas_com_texto": None,
        "paginas_sem_texto": None,
        "respostas": {
            # Formato: [Descrição, Local na Nota, Ordem de Importância, Valor Extraído]
            "[a]": ["Nome da UHE", "Seção 2.3", "1", None],
            "[b]": ["Número da CI", "Seção 1.6", "4", None],
            "[c]": ["Número SEI da CI", "Seção 1.6", "5", None],
            "[d]": ["Pedido (avaliação...)", "Seção 1.6", "6", None],
            "[e]": ["Nome do rio", "Seção 2.3", "2", None],
            "[f]": ["Número do processo", "Seção 1.6", "7", None],
            "[g]": ["Quantidade de folhas do processo", "Seção 1.0", "-", None],
            "[h]": ["Documento de abertura do processo", "Seção 1.6", "8", None],
            "[i]": ["Empreendedor", "Seção 1.6", "3", None],
            "[j]": ["Número da carta do empreendedor com a série", "Seção 1.6", "10", None],
            "[k]": ["Data da carta do empreendedor com a série", "Seção 1.6", "11", None],
            "[l]": ["Número SEI da carta do empreendedor com a série", "Seção 1.6", "12", None],
            "[m]": ["Número da página do processo com a carta do empreendedor com a série", "Seção 2.4", "-", None],
            "[n]": ["Período em anos da série enviada para análise (XXXX a YYYY)", "Seção 2.4", "-", None],
            "[o]": ["Nome da série pelo ONS na planilha excel", "Seção 2.2", "-", None],
            "[p]": ["Período que passa na avaliação (XXXX a YYYY)", "Seção 3", "-", None]
        },
        "pagina_carta": None
    },
    "extracao_outorga": {
        "total_paginas": None,
        "paginas_com_texto": None,
        "paginas_sem_texto": None,
        "paginas": "4-6",
        "nome_aba": "OUTORGA"
    },
    "extracao_PDF_ONS": {
        "paginas": "27-39",
        "nome_aba": "APROVEITAMENTOS"
    },
    "extracao_ONS": {
        "nome_aba": "ONS",
        "identificador_ONS": None,  # A ser definido, caso a caso, automaticamente
        "nome_UHE": None,  # A ser definido, caso a caso, na seção 2.3
        "ano_inicial": 1931, # Valor muda somente quando sai nova consolidação de dados do ONS, uma vez ao ano
        "ano_final": 2024 # Valor muda somente quando sai nova consolidação de dados do ONS, uma vez ao ano
    },
    "extracao_empreendedor": {
        "paginas": None, # Detecção automática das páginas desenvolvida
        "nome_aba": "EMPREENDEDOR"
    },
    "extracao_nota": {
        "nome_aba": "NOTA",
        "nome_doc": None, # A ser definido, caso a caso, automaticamente
    }
}

# Aplicação do perfil de execução às configurações globais do LlamaIndex

print("💡 Configurando splitter de texto...")
Settings.text_splitter = SentenceSplitter(
    chunk_size=PIPELINE_CONTEXT["ambiente"]["chunking"]["size"],
    chunk_overlap=PIPELINE_CONTEXT["ambiente"]["chunking"]["overlap"]
)
print("✅ Text splitter configurado.\n\n💡 Iniciando validações de chaves API Key em Secrets...")

# Validação das chaves API key
validar_chaves_api(userdata)

# Configuração global de modelos (embedding e LLM)
# Configuração de Embeddings
embed_cfg = PIPELINE_CONTEXT["modelos"]["embedding"]
Settings.embed_model = embed_cfg["provider"](**embed_cfg["params"])
print(f"✅ Embedding model: {embed_cfg['params']['model_name']}\n\n💡 Validando dimensões de Embedding...")

# Validação de dimensões de Embeddings pós-instanciação
validar_dimensao_embedding(embed_cfg)

#Configuração de LLM
print("✅ Configurações da vetorização (Embedding) validadas. \n\n💡 Configurando LLM do chatbot...")
Settings.llm = Groq(
    model=PIPELINE_CONTEXT["modelos"]["llm"]["model"]
)
print(f"✅ LLM definido: {PIPELINE_CONTEXT['modelos']['llm']['model']}")

print("✅ Configurações globais finalizadas.\n")

# Validação estrutural e semântica do contexto do pipeline
validar_estrutura_projeto(
    caminho_raiz=PIPELINE_CONTEXT["execucao"]["paths"]["raiz_projeto"],
    subdirs=PIPELINE_CONTEXT["subdirs"]
)

# Validações finais antes da execução do pipeline principal
print("💡 Iniciando validações de contexto do pipeline...")
try:
    validar_contexto_completo(PIPELINE_CONTEXT)
except RuntimeError as e:
    print(f"\n[PARADA DE SEGURANÇA]: {e}")
    raise

###**SEÇÃO 1: Extração de Dados Não-Estruturados e Inteligência Documental**
####Esta seção implementa a camada de visão computacional e processamento de linguagem natural (NLP). É responsável por transformar arquivos PDF (brutos ou imagens) em uma base de conhecimento vetorial consultável, permitindo que o LLM responda perguntas com precisão técnica e citação de fontes.
###**SEÇÃO 1.0**. Preparação do Repositório de Dados Estruturados
####Criação automática da aba de destino na planilha Excel COMPARA. Esta função inicializa o mapeamento de metadados (chaves de [a] a [p]) que receberão os valores extraídos pelo pipeline para a personalização da minuta da nota técnica.
###**SEÇÃO 1.1**. Diagnóstico Documental (Processo e Outorga)
####Varredura inicial de cada página dos PDFs para identificar o "nível de legibilidade". O sistema separa páginas com texto nativo daquelas que são imagens chapadas (escaneamentos), decidindo onde o OCR é obrigatório.
###**SEÇÃO 1.2**. Reconhecimento ótico de caracteres (OCR) e reconstrução documental
####Aplicação do motor Tesseract nas páginas identificadas como imagens. Após o processamento, o código reconstrói o PDF original inserindo uma "camada de texto invisível", gerando os arquivos de base _OCR.pdf.
###**SEÇÃO 1.3**. Seleção de arquivos a processar
####Para a recuperação dedados, são priorizadas as versões processadas por OCR, tanto da outorga quanto do processo.
###**SEÇÃO 1.4**.Parsing
####Utilização do LlamaParse para leitura e pré-processamento de arquivos PDF para a fragmentação do contexto.
###**SEÇÃO 1.5**. Fragmentação (Chunking, Fragmentação (Chunking), vetorização (Embedding) e Indexação FAISS
#### Com uso do SentenceSplitter o texto é dividido em blocos (nodes) semanticamente coerentes, para acelerara recuperação da informação pela vetorização.
####Os blocos de texto são transformados em vetores numéricos de 1024 dimensões, utilizando o modelo multilingual-e5-large. Estes vetores são armazenados no banco de vetores FAISS, permitindo buscas por similaridade em milissegundos.
###**SEÇÃO 1.6**. Query Engine e Chatbot RAG
####Aqui o índice é conectado ao modelo Llama-3.3-70B via Groq, aplicando o System Prompt que obriga a IA a citar páginas e nomes de arquivos em cada resposta. A sessão de chatbot é instanciada,inicialmente, dedicada a perguntas padronizadas e,por fim, é aberta sessão aberta e livre a perguntas aleatórias.
###SEÇÃO 1.7: Transferência de dados das respostas do RAG para a planilha de excel
####As respostas são consolidadas, formatadas com quebra de linha automática e salvas no Excel para análise comparativa.
###**SEÇÃO 1.8**: Extração de Dados por Heurística (Validação)
####Esta seção implementa um motor de extração determinística que serve como contraponto ao RAG. Enquanto a IA busca semântica, este código busca padrões sintáticos e âncoras fixas no texto. Ele valida se as informações críticas (como o Nome da UHE, números SEI e o período da série de vazões) extraídas pela IA coincidem com os padrões rígidos encontrados diretamente nos documentos do processo e da outorga.
####Implementação da Mmúltiplas funções de extração de dados em uma ínica varredura no PDF, otimizando a performance daextração.
####O sistema identifica páginas que contêm tabelas de dadosde vazões ao detectar padrões numéricos (1931 a 2026) e usa essa informação para localizar automaticamente a "Carta do Empreendedor", geralmente situada duas páginas anteriores às tabelas de dados.
####A extração é granulada dos 16 metadados essenciais para personalizar a minuta da nota técnica. Inclui desde identificadores simples como o Nome do Rio até dados complexos como o período em anos da série enviada para análise, garantindo a integridade dos dados que serão levados à planilha de cálculo de apoio à análise da proposição.
####Ao final, os dados são intransferidos para a planilha Excel, permitindo a comparação visual, lado a lado, dos resultados da extração automática (Heurística) e da extração inteligente (RAG).

In [ ]:
# Subseção 1.0 Preparação do Repositório de Dados Estruturados

def criar_aba_nota_em_compara():
    print(f"💡 Preparando a aba {PIPELINE_CONTEXT['extracao_nota']['nome_aba']} em {PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"]}...")

    caminho_excel = PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"]
    respostas = PIPELINE_CONTEXT["extracao_processo"]["respostas"]

    # cada item do dicionário tem [Chave, Descrição, Seção, Ordem]
    tabela_final = []
    for chave, meta in respostas.items():
        tabela_final.append([chave, meta[0], meta[1], meta[2]])

    # Criamos o DataFrame a partir dessa lista processada
    df_nota = pd.DataFrame(tabela_final)

    # 2. Gravação preservando as abas existentes
    try:
        # Verificamos se o arquivo já existe para decidir o modo de escrita. Evita problemas com arquivo aberto. Usa modo append para preservar outros dados caso já existam.
        if os.path.exists(caminho_excel):
            with pd.ExcelWriter(caminho_excel, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
                df_nota.to_excel(writer, sheet_name=PIPELINE_CONTEXT["extracao_nota"]["nome_aba"], index=False, header=False)
        else:
            df_nota.to_excel(caminho_excel, sheet_name=PIPELINE_CONTEXT["extracao_nota"]["nome_aba"], index=False, header=False)

        print(f"✅ Aba '{PIPELINE_CONTEXT['extracao_nota']['nome_aba']}' atualizada com sucesso em: {os.path.basename(caminho_excel)}")

    except Exception as e:
        print(f"❌ Erro ao manipular o Excel: {e}")

criar_aba_nota_em_compara()

In [ ]:
# Subseção 1.1 — Diagnóstico documental dos arquivos a serem processados
print("💡 Iniciando Subseção 1.1 Diagnóstico documental dos arquivos de base...")

def diagnosticar_pdf(caminho_pdf, nome_logico, limite_caracteres, verbose=True):
    """
    Diagnostica um arquivo PDF quanto à existência, número de páginas
    e presença de texto suficiente para evitar OCR.

    Retorna um dicionário com o diagnóstico completo.
    """

    caminho_pdf = Path(caminho_pdf)

    resultado = {
        "nome": nome_logico,
        "arquivo": str(caminho_pdf),
        "total_paginas": 0,
        "paginas_com_texto": [],
        "paginas_sem_texto": [],
        "erro": None
    }

    if not caminho_pdf.exists():
        resultado["erro"] = "Arquivo não encontrado"
        if verbose:
            print(textwrap.fill(
                f"❌ ERRO: O arquivo '{caminho_pdf}' não foi encontrado.", width=100
            ))
        return resultado

    if verbose:
        print(f"\n💡 Diagnóstico documental do arquivo {nome_logico}...")

    try:
        leitor_pdf = PdfReader(str(caminho_pdf))
        resultado["total_paginas"] = len(leitor_pdf.pages)

        if verbose:
            print(textwrap.fill(
                f"✅ O arquivo '{caminho_pdf.name}' tem {resultado['total_paginas']} páginas.",
                width=100
            ))

        for indice, pagina in enumerate(leitor_pdf.pages):
            texto = pagina.extract_text() or ""
            texto_limpo = texto.strip()
            numero_pagina = indice + 1

            if len(texto_limpo) >= limite_caracteres:
                resultado["paginas_com_texto"].append(numero_pagina)
            else:
                resultado["paginas_sem_texto"].append(numero_pagina)

        if verbose:
            print(textwrap.fill("\n--- Relatório de Verificação de Conteúdo ---", width=100))
            print(textwrap.fill(
                f"Critério de Teste: Mínimo de {limite_caracteres} caracteres por página.\n",
                width=100
            ))

            print(textwrap.fill(
                f"## ✅ Páginas que PASSAM no Teste ({len(resultado['paginas_com_texto'])})",
                width=100
            ))
            print(textwrap.fill(
                ", ".join(map(str, resultado["paginas_com_texto"]))
                if resultado["paginas_com_texto"] else "Nenhuma página passa.",
                width=100
            ))

            print(textwrap.fill("\n----------------------------------------------------", width=100))

            print(textwrap.fill(
                f"## ❌ Páginas que NÃO PASSAM no Teste ({len(resultado['paginas_sem_texto'])})",
                width=100
            ))
            print(textwrap.fill(
                ", ".join(map(str, resultado["paginas_sem_texto"]))
                if resultado["paginas_sem_texto"] else "Todas as páginas passam.\n\n",
                width=100
            ))

    except Exception as e:
        resultado["erro"] = str(e)
        if verbose:
            print(textwrap.fill(f"Ocorreu um erro: {e}", width=100))

    return resultado

# Subseção 1.0 — Diagnóstico documental dos arquivos a serem processados
print("✅ Função específica definida com sucesso!")

DOCUMENTOS = {
    "Processo": PIPELINE_CONTEXT["execucao"]["paths"]["pdf_processo"],
    "Outorga": PIPELINE_CONTEXT["execucao"]["paths"]["pdf_outorga"]
}

diagnosticos = {}

for nome, caminho in DOCUMENTOS.items():
    diagnosticos[nome] = diagnosticar_pdf(
        caminho_pdf=caminho,
        nome_logico=nome,
        limite_caracteres=PIPELINE_CONTEXT["ambiente"]["limite_caracteres_legiveis"]
    )
print("\n✅ Diagnóstico documental concluído.")

In [ ]:
#Subseção 1.2 Reconhecimento ótico de caracteres e reconstrução documental
print("💡 Iniciando configurações iniciais da Subseção Reconhecimento Ótico de Caracteres e Reconstrução Documental...")

# Configurações iniciais - definindo funções específicas
def executar_ocr_em_paginas(caminho_pdf, paginas, limite_caracteres, lang='por'):
    """
    Executa OCR somente nas páginas especificadas de um PDF.
    Retorna um dicionário {numero_pagina: texto_ocr}.
    """

    textos_ocr = {}

    if not paginas:
        return textos_ocr

    print(f"Reconhecimento ótico de caracteres nas páginas: {sorted(paginas)}...\n")

    for numero_pagina in sorted(paginas):
        print(f"    -> Processando Página {numero_pagina}...")

        imagens = convert_from_path(
            caminho_pdf,
            dpi=300,
            first_page=numero_pagina,
            last_page=numero_pagina,
            use_pdftocairo=True,
            grayscale=True,
            fmt='png'
        )

        if not imagens:
            print(f"        ❌ Falha ao rasterizar a página {numero_pagina}")
            continue

        imagem = imagens[0]

        texto = pytesseract.image_to_string(imagem, lang=lang).strip()
        textos_ocr[numero_pagina] = texto
        limite_caracteres=PIPELINE_CONTEXT["ambiente"]["limite_caracteres_legiveis"]

        if len(texto) >= limite_caracteres:
            print(f"        ✅ Sucesso! {len(texto)} caracteres detectados.")
        else:
            print(f"        ⚠️ Alerta: Apenas {len(texto)} caracteres detectados.")

    print(f"\n--- OCR concluído: {len(textos_ocr)} páginas processadas ---")
    return textos_ocr

def reconstruir_pdf_com_ocr(caminho_entrada, textos_ocr, paginas_com_ocr, caminho_saida):
    """
    Cria um novo PDF adicionando camada de texto OCR invisível
    às páginas especificadas.
    """

    try:
        doc_original = fitz.open(caminho_entrada)
        doc_novo = fitz.open()

        print(f"\n💡 Iniciando reconstrução de {len(doc_original)} páginas...")

        set_paginas_ocr = set(paginas_com_ocr)

        for i in range(len(doc_original)):
            numero_pagina = i + 1

            doc_novo.insert_pdf(doc_original, from_page=i, to_page=i)
            pagina_nova = doc_novo.load_page(len(doc_novo) - 1)

            if numero_pagina in set_paginas_ocr:
                texto = textos_ocr.get(numero_pagina, "")
                if texto:
                    rect = pagina_nova.rect + (50, 50, -50, -50)
                    pagina_nova.insert_textbox(
                        rect,
                        texto,
                        fontname="helv",
                        fontsize=8,
                        render_mode=3  # invisível
                    )
                    print(f"    [+] OCR inserido na Página {numero_pagina}")

        doc_novo.save(caminho_saida)
        doc_novo.close()
        doc_original.close()

        print(f"\n✅ Reconstrução concluída! PDF salvo em:\n{caminho_saida}")

    except Exception as e:
        print(f"❌ ERRO FATAL na reconstrução do PDF: {e}")
        try: doc_novo.close()
        except: pass
        try: doc_original.close()
        except: pass


def reconstruir_documento_com_ocr(
    nome,
    caminho_pdf,
    paginas_sem_texto,
    limite_caracteres
):
    print(f"💡 Iniciando reconstrução textual do arquivo {nome}...")
    limite_caracteres=PIPELINE_CONTEXT["ambiente"]["limite_caracteres_legiveis"]
    if not paginas_sem_texto:
        print("✅ Nenhuma página sem texto. OCR ignorado.\n")
        return None

    textos_ocr = executar_ocr_em_paginas(
        caminho_pdf=caminho_pdf,
        paginas=paginas_sem_texto,
        limite_caracteres=limite_caracteres
    )

    if not textos_ocr:
        print("⚠️ Nenhum texto OCR gerado.")
        return None

    caminho_saida = caminho_pdf.replace(".pdf", "_OCR.pdf")

    reconstruir_pdf_com_ocr(
        caminho_entrada=caminho_pdf,
        textos_ocr=textos_ocr,
        paginas_com_ocr=paginas_sem_texto,
        caminho_saida=caminho_saida
    )
    nome_arquivo_gerado = os.path.basename(caminho_saida)
    if "PROCESSO" in nome.upper():
        PIPELINE_CONTEXT["execucao"]["paths"]["pdf_processo_ocr"] = caminho_saida
        print(f"✅ Caminho salvo em 'pdf_processo_ocr': {nome_arquivo_gerado}")

    elif "OUTORGA" in nome.upper():
        PIPELINE_CONTEXT["execucao"]["paths"]["pdf_outorga_ocr"] = caminho_saida
        print(f"✅ Caminho salvo em 'pdf_outorga_ocr': {nome_arquivo_gerado}")

    return caminho_saida

#Aplicação das funções específicas
print("✅ Ambiente configurado com sucesso!\n")
DOCUMENTOS_OCR = {}

for nome, info in diagnosticos.items():
    DOCUMENTOS_OCR[nome] = {
        "arquivo": info["arquivo"],
        "paginas_sem_texto": info["paginas_sem_texto"]
    }
arquivos_ocr = {}

for nome, info in DOCUMENTOS_OCR.items():
    arquivos_ocr[nome] = reconstruir_documento_com_ocr(
        nome=nome,
        caminho_pdf=info["arquivo"],
        paginas_sem_texto=info["paginas_sem_texto"],
        limite_caracteres=PIPELINE_CONTEXT["ambiente"]["limite_caracteres_legiveis"]
    )

print("✅ Reconstrução textual concluída para todos os documentos.")

In [ ]:
#Subseção 1.3 Seleção de arquivos a processar

def selecionar_pdf_com_prioridade_ocr(caminho_pdf):
    """
    Retorna o caminho do PDF priorizando a versão OCR (_OCR.pdf), se existir.
    Caso contrário, retorna o arquivo original.
    """
    caminho = Path(caminho_pdf)

    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

    caminho_ocr = caminho.with_name(caminho.stem + "_OCR" + caminho.suffix)

    if caminho_ocr.exists():
        print(f" -> Versão OCR priorizada: {caminho_ocr.name}")
        return str(caminho_ocr)

    print(f" -> Versão OCR não encontrada. Usando original: {caminho.name}")
    return str(caminho)

print("\n💡 Iniciando Subseção 1.5 Seleção de arquivos para processamento...")

processo_selecionado = selecionar_pdf_com_prioridade_ocr(ARQUIVO_PROCESSO)
try:
    outorga_selecionada  = selecionar_pdf_com_prioridade_ocr(ARQUIVO_OUTORGA)
except FileNotFoundError:
    outorga_selecionada = None

print("\nIniciando a validação da lista de documentos para processamento...")

documentos_finais = [processo_selecionado, outorga_selecionada]

if len(documentos_finais) < 2:
    raise FileNotFoundError(
        "FALHA NA SELEÇÃO: Não foi possível encontrar os dois arquivos essenciais ('processo' e 'outorga')."
    )

print("Arquivos encontrados na lista documentos_finais:")
for arquivo in documentos_finais:
    print(f"- {arquivo}")

nomes = ", ".join([os.path.basename(d) for d in documentos_finais if d])
print(f"✅ Documentos Selecionados: {nomes}")

In [ ]:
# Subseção 1.4 PARSING
print("💡 Iniciando Subseção 1.4 Avaliação preliminar...")
try:
    with open(PIPELINE_CONTEXT["execucao"]["paths"]["persistencia_drive"], "rb") as f:
        docs = pickle.load(f)
    print(f"✅ Os documentos já passaram pelo Parser. Total de documentos carregados: {len(docs)}\n\n")

except (FileNotFoundError, EOFError, pickle.UnpicklingError):
    print("Os documentos ainda não foram analisados pelo Parser.\n\n💡 Configurando o parser...")

    # 1. Configuração do parser (LlamaParse)
    parser = LlamaParse(
        api_key=os.environ.get("LLAMA_CLOUD_API_KEY"),
        disable_ocr=True,
        page_prefix="--- PÁGINA {pageNumber} ---\n",
        page_suffix="\n--- FIM PÁGINA {pageNumber} ---",
        user_prompt="Extract all tables in markdown format. Do not summarize tables.",
        disable_image_extraction=True,
        spreadsheet_extract_sub_tables=True,
        output_tables_as_HTML=False,
        num_workers=1,
        show_progress=True,
        preserve_layout_alignment_across_pages=True
    )
    max_concurrent = PIPELINE_CONTEXT["query_engine"]["similarity_top_k"]
    semaphore = asyncio.Semaphore(max_concurrent)
    print("✅ Parser e semaphore configurados com sucesso!\n\n💡 Iniciando a análise dos documentos pelo Parser...")
    async def parse_single_file(parser, file_path, semaphore):
        async with semaphore:
            return await parser.aparse(file_path)

    pdf_files = documentos_finais
    if not pdf_files:
        raise RuntimeError("❌ A lista documentos_finais está vazia.")
    tasks = [parse_single_file(parser, f, semaphore) for f in pdf_files]
    results = await asyncio.gather(*tasks)

    docs = []
    for file_path, parse_result in zip(pdf_files, results):
        if isinstance(parse_result, list):
            for llama_doc in parse_result:
                llama_doc.metadata.update({
                    "file_name": os.path.basename(str(file_path)),
                })
                docs.append(llama_doc)
        else:
            for page_idx, page in enumerate(parse_result.pages, start=1):
                docs.append(Document(
                    text=getattr(page, "markdown", None) or page.text or "",
                    metadata={
                        "file_name": os.path.basename(str(file_path)),
                        "page_number": page_idx,
                    }
                ))

    if not docs:
        raise RuntimeError("❌ Nenhuma página extraída pelo parser.")

    # 4. Salvar os documentos processados
    print("✅ Documentos analisados com sucesso!\n\n💡 Salvando os documentos processados em arquivo pickle no drive...")
    caminho_local_pkl = PIPELINE_CONTEXT["execucao"]["paths"]["persistencia_local"]
    caminho_drive_pkl = PIPELINE_CONTEXT["execucao"]["paths"]["persistencia_drive"]
    try:
        tmp_local = caminho_local_pkl + ".tmp"
        with open(tmp_local, "wb") as f:
            pickle.dump(docs, f)

        os.replace(tmp_local, caminho_local_pkl)
        print(f"✅ Dados persistidos localmente no HD Virtual.")
        os.makedirs(os.path.dirname(caminho_drive_pkl), exist_ok=True)
        shutil.copy2(caminho_local_pkl, caminho_drive_pkl)

        print(f"✅ Sucesso! Dados transferidos para o Drive em: {caminho_drive_pkl}")
        print("\n💡 Iniciando verificações finais do parsing...")
    except Exception as e:
        print(f"❌ Erro ao salvar o arquivo: {e}")

    # 5. Verificações finais
    total_paginas = len(docs)
    print(f"✅ Total de páginas extraídas: {total_paginas}")

    # Distribuição por arquivo
    dist_por_arquivo = Counter(d.metadata["file_name"] for d in docs)

    print("✅ Páginas analisadas por arquivo:")
    for arquivo, qtd in dist_por_arquivo.items():
        print(f"   • {arquivo}: {qtd} páginas")

    if total_paginas != sum(dist_por_arquivo.values()):
        print("⚠️ Atenção: inconsistência na contagem de páginas!")
    else:
        print("✅ Contagem consistente. Parsing validado com sucesso.\n")

In [ ]:
#Subseção 1.5 Fragmentação (Chunking), vetorização (Embedding) e Indexação
# Em um sistema RAG, o parsing (leitura e extração do conteúdo bruto dos arquivos) precede o chunking (divisão em partes menores), o embedding (transformação em vetores) e a indexação (armazenamento para busca eficiente)
# Parsing não é apenas dividir um conteúdo em páginas; é o processo de ler, interpretar e estruturar o conteúdo bruto de um arquivo (como PDF, DOCX, HTML) em objetos manipuláveis pelo sistema, extraindo texto, metadados e, se necessário, imagens ou tabelas fonte.
# Dividir manualmente um texto longo em arquivos PDF separados para cada página não é parsing no sentido técnico; é apenas uma divisão física. O parsing envolve transformar o conteúdo desses arquivos em uma estrutura compreendida pelo pipeline de processamento.

# 1. Procurando por arquivos analisados na seção anterior (Parsing)
print("\n💡 Procurando por dados persistidos na seção anterior (Parsing)...")
try:
    with open(PIPELINE_CONTEXT["execucao"]["paths"]["persistencia_drive"], "rb") as f:
        docs = pickle.load(f)
    print(f"✅ Arquivo carregado: {len(docs)} documentos encontrados.\n\n💡 Iniciando a fragmentação...")
except Exception as e:
    print(f"❌ Erro ao carregar o arquivo para chunking: {e}")
    raise

# 2. Configurando o Splitter com os dados do PIPELINE_CONTEXT
splitter = SentenceSplitter(
    chunk_size=PIPELINE_CONTEXT["ambiente"]["chunking"]["size"],
    chunk_overlap=PIPELINE_CONTEXT["ambiente"]["chunking"]["overlap"]
)

# 3. Aplicando o splitter para a fragmentação (criação dos nodes)
nodes = splitter.get_nodes_from_documents(docs)

print(f"✅ Fragmentação (Chunking) concluída: {len(nodes)} nós criados.\n\n💡 Verificando a compatibilidade entre a vetorização (Embedding) com o parâmetro 'd' do Índice FAISS)... ")

embed_config = PIPELINE_CONTEXT["modelos"]["embedding"]
embed_model = embed_config["provider"](
    **embed_config["params"]
)

# 4. Configurando o Indexador FAISS (Facebook AI Similarity Search) para a criação de vetores indexados
d = embed_config["embed_dim"]

# 5. Verificando a compatibilidade do Embedding com o parâmetro 'd' do índice FAISS
embedding_teste = embed_model.get_text_embedding("teste")
print("Dimensão do vetor gerado em teste de vetorização (Embedding):", len(embedding_teste))
print("Dimensão configurada para o banco de vetores FAISS (d):", d)
assert len(embedding_teste) == d, f"Erro: o vetor gerado no Embedding tem comprimento {len(embedding_teste)}, mas o banco de vetores FAISS espera por vetores com {d}"
print(f"✅ Dimensão do vetor gerado na vetorização (Embedding) de {d} unidades é compatível com o tamanho esperado pelo banco de vetores FAISS.\n\n💡 Verificando o tamanho dos vetores individualmente... ")

# Verificação do tamanho dos vetores gerados em relação às configurações do banco de vetores FAISS

print("Compatibilidade dos vetores verificada individualmente. Vetores compatíveis com o FAISS.\n\n💡 Iniciando a indexação dos vetores, criação do índice... ")
faiss_index = faiss.IndexFlatL2(d)

# 6. Criando o banco de vetores (Vector Store)
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 7. Indexando os vetores da fragmentação (dos nodes)
index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    show_progress=True,
    embed_model=embed_model
)
print(f"✅ Vetores indexados com sucesso!\n\n💡 Persistindo o índice localmente em: {PIPELINE_CONTEXT["execucao"]["paths"]["persistencia_local"]}...")
diretorio_persist = PIPELINE_CONTEXT['execucao']['paths']['persistencia_local']

diretorio_persist = PIPELINE_CONTEXT['execucao']['paths']['persistencia_local']

# 1. Limpeza Inteligente: Deleta seja arquivo ou pasta
if os.path.exists(diretorio_persist):
    if os.path.isdir(diretorio_persist):
        print(f"♻️ Removendo diretório antigo: {diretorio_persist}")
        shutil.rmtree(diretorio_persist)
    else:
        print(f"♻️ Removendo qualquer arquivo conflitante antigo: {diretorio_persist}")
        os.remove(diretorio_persist)

# 2. Agora criamos do zero, com segurança
os.makedirs(diretorio_persist, exist_ok=True)

# 3. Persistência
index.storage_context.persist(persist_dir=diretorio_persist)

print(f"📂 Estrutura de arquivos do Índice no HD Virtual: {diretorio_persist}")

diretorio_persist = PIPELINE_CONTEXT['execucao']['paths']['persistencia_local']
os.makedirs(diretorio_persist, exist_ok=True)
index.storage_context.persist(persist_dir=diretorio_persist)

print(f"📂 Estrutura de arquivos do Índice no HD Virtual: {diretorio_persist}")
for arquivo in os.listdir(PIPELINE_CONTEXT["execucao"]["paths"]["persistencia_local"]):
    print(f"    ├── {arquivo}")
print("✅ Índice persistido no HD virtual.\n")

# 9. Compactação local (Ainda no HD Virtual)
origem = PIPELINE_CONTEXT["execucao"]["paths"]["persistencia_local"]
destino_zip = PIPELINE_CONTEXT["execucao"]["paths"]["compactado_local"]
print(f"💡 Compactando o índice em: {destino_zip}")
try:
    with tarfile.open(destino_zip, "w:gz") as tar:
        tar.add(origem, arcname=os.path.basename(origem))
    print("✅ Compactação local concluída.")
except Exception as e:
    print(f"❌ Erro na compactação: {e}")

# 10. Transferência para o Drive (Copiando o .tar.gz)
origem_zip = PIPELINE_CONTEXT["execucao"]["paths"]["compactado_local"]
destino_drive = PIPELINE_CONTEXT["execucao"]["paths"]["compactado_drive"]
print(f"💡 Transferindo compactado para o Drive...")
try:
    shutil.copy2(origem_zip, destino_drive)
    print(f"✅ Sucesso! Índice transferido para: {destino_drive}")
except Exception as e:
    print(f"❌ Erro na transferência do arquivo compactado do HD local para o Drive: {e}")

# 11. Descompactação no Drive para uso do Query Engine
caminho_zip_drive = PIPELINE_CONTEXT["execucao"]["paths"]["compactado_drive"]
pasta_destino_drive = PIPELINE_CONTEXT["execucao"]["paths"]["raiz_projeto"]

print(f"💡 Iniciando descompactação no Drive...")
print(f"📂 Origem: {caminho_zip_drive}")
print(f"🎯 Destino: {pasta_destino_drive}")

try:
    with tarfile.open(caminho_zip_drive, "r:gz") as tar:
        tar.extractall(path=pasta_destino_drive)
    print(f"✅ Sucesso! O índice foi extraído e está disponível na pasta raiz do Drive.")

    # 3. Verificação visual do que foi extraído
    pasta_extraida = os.path.join(pasta_destino_drive, os.path.basename(PIPELINE_CONTEXT["execucao"]["paths"]["persistencia_local"]))

    if os.path.exists(pasta_extraida):
        print(f"📁 Conteúdo da pasta extraída ({os.path.basename(pasta_extraida)}):")
        for item in os.listdir(pasta_extraida):
            print(f"   └── {item}")

except FileNotFoundError:
    print(f"❌ Erro: O arquivo compactado não foi encontrado no Drive. Verifique se o caminho está correto.")
except Exception as e:
    print(f"❌ Erro durante a descompactação: {e}")

# 12. Verificação Final de Disponibilidade no DRIVE

caminho_indice_drive = os.path.join(
    PIPELINE_CONTEXT["execucao"]["paths"]["raiz_projeto"],
    os.path.basename(PIPELINE_CONTEXT["execucao"]["paths"]["persistencia_local"])
)

print(f"📂 Verificando arquivos no DRIVE em: {caminho_indice_drive}")

if os.path.exists(caminho_indice_drive):
    for raiz, diretorios, arquivos in os.walk(caminho_indice_drive):
        for arquivo in arquivos:
            print(f"    ├── {arquivo}")

    msg = ("O arquivo 'default__vector_store.json' é o arquivo binário do índice FAISS, não é um JSON. Portanto, este arquivo contém a estrutura do índice."
    )

    print("\n" + textwrap.fill(msg, width=100))
    print("✅ Índice pronto para o Query Engine no Drive.")
else:
    # Correção das aspas para evitar SyntaxError
    print(f"❌ ERRO: Diretório não localizado no Drive!")
    print(f"Caminho esperado: {caminho_indice_drive}")

In [ ]:
# Seção 1.6 Query Engine e Chatbot RAG
# 1. Restauração do Vector Store
print("💡 Restaurando o vector store...")

try:
    caminho_indice_drive = os.path.join(PIPELINE_CONTEXT['execucao']['paths']['raiz_projeto'], 'faiss_index_storage')
    vector_store = FaissVectorStore.from_persist_dir(caminho_indice_drive)
except Exception as e:
    print(f"❌ Erro ao restaurar o vector store: {e}")
    raise

# 2. Definição do Storage Context
print("✅ Vector store restaurado.\n\n💡 Definindo o storage context...")
try:
    storage_context = StorageContext.from_defaults(
        vector_store=vector_store,
        persist_dir=caminho_indice_drive
    )
except Exception as e:
    print(f"❌ Erro ao definir o storage context: {e}")
    raise

# 3. Carregamento do índice persistido no Drive para consulta
print("✅ Storage context definido.\n\n💡 Carregando o índice persistido para consulta...")
try:
    index = load_index_from_storage(storage_context)
except Exception as e:
    print(f"❌ Erro ao recuperar o índice: {e}")
    raise

# 4. Criação do Chatbot para consultas à documentação
print("✅ Índice carregado para consulta.\n\n💡 Criando o chatbot para consulta...")

from llama_index.core import PromptTemplate

# 5. Preparar o Prompt Template usando o seu system_prompt do contexto
text_qa_template_str = (
    PIPELINE_CONTEXT["query_engine"]["system_prompt"] +
    "\n\nCONTEXTO:\n{context_str}\n\n"
    "PERGUNTA: {query_str}\n\n"
    "RESPOSTA:"
)
text_qa_template = PromptTemplate(text_qa_template_str)

# 6. Criar o Query Engine direcionado
query_engine = index.as_query_engine(
    similarity_top_k=PIPELINE_CONTEXT["query_engine"]["similarity_top_k"],
    text_qa_template=text_qa_template,
    response_mode="compact"
)
print("✅ Query Engine configurado com as diretrizes do PIPELINE_CONTEXT.\n\n🤖 Chatbot dedicado às perguntas padrão do pipeline...")

# 7. Parte A: PROCESSAMENTO DE PERGUNTAS PADRÃO
respostas_consolidadas = []
for pergunta in PIPELINE_CONTEXT["query_engine"]["perguntas_padrao"]:
    try:
        resultado = query_engine.query(pergunta)
        texto_ia = resultado.response
        lista_paginas = [str(n.metadata.get("page_label") or n.metadata.get("page_number", "?")) for n in resultado.source_nodes]
        fontes_paginas = ", ".join(sorted(set(lista_paginas)))
        trechos_originais = " | ".join([node.text[:200] + "..." for node in resultado.source_nodes])
        print(f"💡 Pergunta: {textwrap.fill(pergunta, width=100)}\n✅ Resposta: {textwrap.fill(texto_ia, width=100)}")
        respostas_consolidadas.append({
            "Pergunta": pergunta,
            "Resposta_Extraida": texto_ia,
            "Paginas_Fonte": fontes_paginas,
            "Contexto_Original": trechos_originais
        })
        time.sleep(3)
    except Exception as e:
        print(f"❌ Erro: {e}")
        respostas_consolidadas.append({
            "Pergunta": pergunta,
            "Resposta_Extraida": "ERRO NA EXTRAÇÃO",
            "Paginas_Fonte": "-",
            "Contexto_Original": "-"
        })
    print("-" * 50)

# 8. Parte B: SEÇÃO ABERTA PARA PERGUNTAS (CHAT)
print("\n💬 Chatbot disponível para seção aberta de perguntas. Digite sua pergunta sobre a documentação (ou 'sair' para encerrar) e tecle ENTER:")

while True:
    pergunta_usuario = input("\nSua pergunta: ")

    if pergunta_usuario.lower() in ['sair', 'exit', 'quit', 'parar']:
        print("👋 Encerrando seção de consulta. Até logo!")
        break

    if not pergunta_usuario.strip():
        continue

    print("🔍 Pesquisando no índice...")
    try:
        resultado_chat = query_engine.query(pergunta_usuario)
        texto_ia_chat = resultado_chat.response
        paginas_chat = [str(n.metadata.get("page_label") or n.metadata.get("page_number", "?")) for n in resultado_chat.source_nodes]
        fontes_unicas = ", ".join(sorted(set(paginas_chat)))
        print(f"\n🤖 Resposta:\n{textwrap.fill(texto_ia_chat, width=100)}")
        print(f"📍 Fontes: Páginas {fontes_unicas}")
        respostas_consolidadas.append({
            "Pergunta": f"[CHAT] {pergunta_usuario}",
            "Resposta_Extraida": texto_ia_chat,
            "Paginas_Fonte": fontes_unicas,
            "Contexto_Original": "Consulta manual via chat"
        })

    except Exception as e:
        print(f"❌ Erro ao processar pergunta: {e}")

# 9. Persistência e formatação
print("\n\n💡 Iniciando persistência e formatação no arquivo COMPARA no Drive...")

try:
    df = pd.DataFrame(respostas_consolidadas)
    df['Data_Analise'] = datetime.now().strftime("%d/%m/%Y %H:%M")
    caminho_arquivo = PIPELINE_CONTEXT["query_engine"]["persistencia"]
    aba_destino = PIPELINE_CONTEXT["query_engine"]["nome_aba"]
    if os.path.exists(caminho_arquivo):
        with pd.ExcelWriter(caminho_arquivo, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
            df.to_excel(writer, sheet_name=aba_destino, index=False)
    else:
        with pd.ExcelWriter(caminho_arquivo, engine="openpyxl", mode="w") as writer:
            df.to_excel(writer, sheet_name=aba_destino, index=False)

    # Formatação da planilha Respostas_RAG gerada em COMPARA
    wb = load_workbook(caminho_arquivo)
    ws = wb[aba_destino]
    column_widths = {
        'A': 43,  # ~300px
        'B': 43,  # ~300px
        'C': 16,  # ~110px
        'D': 43,  # ~300px
        'E': 17   # ~120px
    }
    for col, width in column_widths.items():
        ws.column_dimensions[col].width = width
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical='top', horizontal='left')
    wb.save(caminho_arquivo)
    print(f"✅ Respostas persistidas no Drive!")
    print(f"   📄 Arquivo: {caminho_arquivo} | Aba: {aba_destino}")

except PermissionError:
    print(f"❌ ERRO: O arquivo Excel está aberto ou bloqueado pelo sistema.")
except Exception as e:
    print(f"❌ Erro inesperado: {e}")

In [ ]:
#Subseção 1.7 Transferência de dados das respostas do RAG para a planilha de excel
print("💡 Iniciando transferência de dados do RAG para a minuta da Nota Técnica...")

def transferir_dados_especificos(caminho_excel):
    if not os.path.exists(caminho_excel):
        print(f"Erro: O arquivo {caminho_excel} não foi encontrado.")
        return
    wb = openpyxl.load_workbook(caminho_excel)
    nome_aba_nota = PIPELINE_CONTEXT["extracao_nota"]["nome_aba"]
    nome_aba_rag = PIPELINE_CONTEXT["query_engine"]["nome_aba"]

    if nome_aba_rag not in wb.sheetnames or nome_aba_nota not in wb.sheetnames:
        print(f"Erro: As abas '{nome_aba_rag}' ou '{nome_aba_nota}' não existem.")
        return
    ws_rag = wb[nome_aba_rag]
    ws_nota = wb[nome_aba_nota]
    # Mapeamento: Chave é a linha na aba NOTA (destino),
    # Valor é a linha na aba RAG_respostas (origem)
    # A coluna B em RAG_respostas é a 'Resposta_Extraida' (índice 2)
    respostas_contexto = PIPELINE_CONTEXT["extracao_processo"]["respostas"]
    mapeamento_tags = {
        "[a]": [1, 2],  "[e]": [5, 3],  "[i]": [9, 4],
        "[b]": [2, 5],  "[c]": [3, 6],  "[d]": [4, 7],
        "[f]": [6, 8],  "[h]": [8, 9],  "[j]": [10, 11],
        "[k]": [11, 12], "[l]": [12, 13]
    }
    for tag, linhas in mapeamento_tags.items():
        linha_nota, linha_rag = linhas
        valor_rag = ws_rag.cell(row=linha_rag, column=2).value
        if valor_rag is not None:
            ws_nota.cell(row=linha_nota, column=5).value = valor_rag
            if tag in respostas_contexto:
                respostas_contexto[tag][3] = valor_rag
            print(f"Tag {tag}: Aba {nome_aba_nota} L{linha_nota} atualizada.")
    wb.save(caminho_excel)
    print(f"✅ Sincronização concluída na aba {nome_aba_nota}.")

def limpar_coluna_nota(caminho_arquivo):
    nome_aba_nota = PIPELINE_CONTEXT["extracao_nota"]["nome_aba"]
    print(f"🧹 Iniciando limpeza na aba '{nome_aba_nota}'...")
    try:
        wb = openpyxl.load_workbook(caminho_arquivo)
        if nome_aba_nota not in wb.sheetnames:
            print(f"❌ Aba '{nome_aba_nota}' não encontrada.")
            return
        ws = wb[nome_aba_nota]
        respostas_contexto = PIPELINE_CONTEXT["extracao_processo"]["respostas"]
        linha_para_tag = {1: "[a]", 2: "[b]", 3: "[c]", 4: "[d]", 5: "[e]", 6: "[f]",
                          8: "[h]", 9: "[i]", 10: "[j]", 11: "[k]", 12: "[l]"}
        frase_erro = "Não localizei a resposta."
        for row in range(1, 14):
            celula = ws.cell(row=row, column=5)
            texto = str(celula.value) if celula.value else ""
            if texto:
                if frase_erro.lower() in texto.lower():
                    texto_limpo = ""
                else:
                    texto_limpo = re.sub(r'\s*\(.*?\)', '', texto).strip()

                celula.value = texto_limpo
                tag = linha_para_tag.get(row)
                if tag and tag in respostas_contexto:
                    respostas_contexto[tag][3] = texto_limpo
        wb.save(caminho_arquivo)
        print(f"✅ Limpeza e sincronização finalizadas na aba {nome_aba_nota}.")
    except Exception as e:
        print(f"❌ Erro na limpeza: {e}")
# Execução
if __name__ == "__main__":
    transferir_dados_especificos(COMPARA)
    limpar_coluna_nota(COMPARA)

In [ ]:
# SEÇÃO 1.8 Procura pelos dados ancorada em termos fixos (Heurística)

def processar_documento_fiel_otimizado(caminho_pdf):
    if not os.path.exists(caminho_pdf):
        print(f"❌ Erro: Arquivo não encontrado em {caminho_pdf}")
        return None

    leitor = PdfReader(caminho_pdf)
    res = {}
    total_paginas = len(leitor.pages)

    # Objetos para busca de tabelas
    table_reader = PDFTableReader()
    paginas_tabelas = []

    # Alvos de busca
    frase_alvo_ci = "solicitamos avaliação desta SHE quanto à atualização da série de vazões que consta da Outorga"
    frase2_alvo_ci = "solicitar da SHE a avaliação da série de vazões afluentes"
    frase3_alvo_ci = "consultamos essa Superintendência de Estudos Hídricos e Socioeconômicos - SHE sobre a série de vazões médias mensais afluente ao aproveitamento"
    p1 = re.sub(r'\s+', r'\\s+', frase_alvo_ci)
    p2 = re.sub(r'\s+', r'\\s+', frase2_alvo_ci)
    p3 = re.sub(r'\s+', r'\\s+', frase3_alvo_ci)
    padrao_alvo_ci = f"({p1}|{p2}|{p3})"
    texto_da_pagina_alvo_ci = ""
    fragmentos_ci = []

    print(f"🚀 Iniciando Processamento Unificado: {os.path.basename(caminho_pdf)}")

    # LOOP ÚNICO: Percorre o PDF uma vez
    for i in tqdm(range(total_paginas), desc="Analisando Documento"):
        pagina = leitor.pages[i]
        num_pagina = i
        texto_puro = pagina.extract_text()

        if not texto_puro: continue

        # 1. BUSCA PELA CI (Se ainda não encontrou)
        if not fragmentos_ci and re.search(padrao_alvo_ci, texto_puro, re.IGNORECASE | re.DOTALL):
            print(f"\n🎯 CI encontrada na página {num_pagina+1}")
            texto_da_pagina_alvo_ci = texto_puro
            fragmentos_ci = texto_puro.split('\n')

        # 2. BUSCA POR TABELAS DE VAZÕES (Lógica de anos sequenciais)
        # Usamos regex no texto puro primeiro para decidir se vale chamar o table_reader (ganho de performance)
        primeiras_partes = [re.split(r'[,\s\t]', linha.strip())[0] for linha in texto_puro.split('\n')[:20]]
        anos_potenciais = [re.sub(r'\D', '', p) for p in primeiras_partes if len(re.sub(r'\D', '', p)) == 4]

        if len(anos_potenciais) >= 3: # Filtro rápido antes de processar tabela
            try:
                # Se o texto sugere uma tabela, validamos com o carregador de tabelas
                docs = table_reader.load_data(file=Path(caminho_pdf), pages=str(num_pagina))
                for doc in docs:
                    linhas_tab = doc.text.strip().split("\n")
                    anos_na_pagina = []
                    for lin in linhas_tab:
                        p_parte = re.split(r'[,\s\t]', lin.strip())[0]
                        so_nums = re.sub(r'\D', '', p_parte)
                        if so_nums and 1930 <= int(so_nums) <= 2026:
                            anos_na_pagina.append(int(so_nums))

                    if len(anos_na_pagina) >= 3:
                        paginas_tabelas.append(num_pagina)
                        break
            except: pass

    # --- PÓS-PROCESSAMENTO (Lógica de Extração) ---

    # Define página da carta baseado na primeira tabela encontrada
    if paginas_tabelas:
        paginas_tabelas.sort()
        PIPELINE_CONTEXT["extracao_processo"]["pagina_carta"] = paginas_tabelas[0] - 2
        print(f"📈 Tabelas detectadas iniciando na pág: {paginas_tabelas[0]}")

    if not fragmentos_ci:
        print("❌ Erro: CI não localizada no documento.")
        return None

    mapa = dict(enumerate(fragmentos_ci))

    # [a] UHE
    for idx, frag in mapa.items():
        if "UHE" in frag:
            texto_comp = (frag + " " + mapa.get(idx + 1, "")).strip()
            match = re.search(r"UHE\s+([A-ZÁ-ÝÀ-Ù][\w\s\(\)./,-]+)", texto_comp)
            res["[a]"] = f"UHE {match.group(1).strip()}".split('.')[0].split(',')[0].strip() if match else texto_comp
            print(f"✅ [a]: {res['[a]']}")
            break

    # [b] e [c] Comunicação Interna
    for idx, frag in mapa.items():
        if "COMUNICAÇÃO INTERNA" in frag:
            res["[b]"] = frag.strip()[23:]
            res["[c]"] = mapa.get(idx + 1, "").strip()[13:]
            break

    # [d] Pedido e [n][p] Anos
    for idx, frag in mapa.items():
        termo1_busca = "solicitamos"
        termo2_busca = "vimos solicitar da SHE a"
        termo3_busca = "consultamos essa Superintendência de Estudos Hídricos e Socioeconômicos - SHE sobre a"
        if termo1_busca in frag or termo2_busca in frag:
            texto_comp = (frag + " " + mapa.get(idx + 1, "")).strip()
            match = re.search(r"(?:solicitamos|vimos solicitar da SHE a|consultamos essa Superintendência de Estudos Hídricos e Socioeconômicos - SHE sobre a)\s+(.*?)\.", texto_comp, re.IGNORECASE | re.DOTALL)
            res["[d]"] = match.group(1).strip() if match else texto_comp

            # Extração de anos integrada
            anos = sorted(list(set(re.findall(r"\b(19\d{2}|20[0-2]\d)\b", res["[d]"]))))
            if anos:
                res["[n]"] = f"{min(anos)} a {max(anos)}"
                res["[p]"] = res["[n]"]
            break

    # [e] Rio e [i] Empreendedor
    m_e = re.search(r"rio\s+([A-Z][a-zà-ÿ]+(?:\s+[A-Z][a-zà-ÿ]+)*)", texto_da_pagina_alvo_ci)
    if m_e: res["[e]"] = f"rio {m_e.group(1)}"

    m_i = re.search(r"(?:protocolado por|encaminhado pela)\s+(.*?)(?=,)", texto_da_pagina_alvo_ci, re.IGNORECASE | re.DOTALL)
    if m_i: res["[i]"] = m_i.group(1).replace('\n', ' ').strip()

    # [f] Referência e [g] Total Páginas
    for idx, frag in mapa.items():
        termo1_busca = "Referência:"
        termo2_busca= "Processo n"
        if termo1_busca in frag or termo2_busca in frag:
            res["[f]"] = frag.strip()[12:]
            break
    res["[g]"] = total_paginas

    # [h] Requerimento (Página 3 fixa)
    try:
        texto_p3 = leitor.pages[2].extract_text().split('\n')
        for idx, seg in enumerate(texto_p3):
            if "REQUERIMENTO DE OUTORGA" in seg:
                res["[h]"] = f"{seg} do {texto_p3[idx+1]} {texto_p3[idx+2]}".lower().strip()
                break
    except: res["[h]"] = "Não localizado na pág 3"

    # [j], [k], [m] Carta do Empreendedor
    pg_carta_idx = PIPELINE_CONTEXT["extracao_processo"].get("pagina_carta")
    if pg_carta_idx is not None:
        pg_carta_idx = int(pg_carta_idx)
        if pg_carta_idx < total_paginas:
            texto_carta = leitor.pages[pg_carta_idx].extract_text().split('\n')
            res["[j]"] = texto_carta[1].strip() if len(texto_carta) > 1 else ""
            match_dinamico = next((re.search(r'\d.*', d) for d in texto_carta if re.search(r'\d.*', d)), None)
            res["[k]"] = match_dinamico.group().strip() if match_dinamico else ""
            res["[m]"] = pg_carta_idx + 1
        else:
            res["[m]"] = "Verificar manualmente"

        res["[o]"] = PIPELINE_CONTEXT["extracao_ONS"].get("identificador_ONS", "N/A")

        # Injeção final no PIPELINE_CONTEXT preservando estrutura
        contexto_res = PIPELINE_CONTEXT["extracao_processo"]["respostas"]
        for chave in contexto_res.keys():
                contexto_res[chave][3] = res.get(chave, "")
        if not res.get(chave):
            nome_campo = contexto_res[chave][0]
            print(f"⚠️ Chave [{chave}] {nome_campo} ficou vazia.")

    return res

# 1. Definição do caminho (Permanece igual)
caminho_ocr = os.path.join(RAIZ, 'processo_OCR.pdf')
caminho_original = PIPELINE_CONTEXT["execucao"]["paths"]["pdf_processo"]
CAMINHO_FINAL_PROCESSO = caminho_ocr if os.path.exists(caminho_ocr) else caminho_original

# 2. EXECUÇÃO OTIMIZADA (A grande mudança está aqui)
# Chamamos apenas UMA função que faz o trabalho das duas anteriores de uma vez
processar_documento_fiel_otimizado(CAMINHO_FINAL_PROCESSO)

# 3. Fallback Manual (Caso a função unificada não tenha achado a página da carta)
if PIPELINE_CONTEXT["extracao_processo"].get("pagina_carta") is None:
    print("\n" + "!"*60)
    print("⚠️  AVISO: Não conseguimos localizar a tabela ou a carta automaticamente.")
    val = input("👉 Digite a página da carta do empreendedor ou 'pular': ").strip()
    PIPELINE_CONTEXT["extracao_processo"]["pagina_carta"] = val
    # Se o usuário digitou uma página agora, rodamos uma extração rápida só para as chaves da carta
    if val.isdigit():
        # (Opcional) Você pode criar uma mini função só para j, k, m aqui se quiser
        pass
    print("!"*60 + "\n")

# 4. Visualização e Gravação (Permanece igual, pois a estrutura de dados foi preservada)
print(f"{'CHAVE':<7} | {'VALOR EXTRAÍDO'}")
print("-" * 30)
res = PIPELINE_CONTEXT["extracao_processo"]["respostas"]

for chave, lista_meta in res.items():
    item, valor = (lista_meta[0], lista_meta[3]) if isinstance(lista_meta, list) else (None, lista_meta)
    print(f"{chave:<5} | {item} é {valor}")

# 5. Gravação no Excel
caminho_excel = PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"]
mapeamento_linhas = {
    "[a]": 1, "[b]": 2, "[c]": 3, "[d]": 4, "[e]": 5,
    "[f]": 6, "[g]": 7, "[h]": 8, "[i]": 9, "[j]": 10, "[k]": 11,
    "[l]": 12, "[m]": 13, "[n]": 14, "[o]": 15, "[p]": 16
}

for chave, lista_meta in res.items():
    if chave in mapeamento_linhas:
        linha_excel = mapeamento_linhas[chave]
        valor_para_excel = lista_meta[3] if isinstance(lista_meta, list) else lista_meta
        if valor_para_excel is None: valor_para_excel = ""
        gravar_valor_detectado(caminho_excel, valor_para_excel, linha_excel, 6)

###**SEÇÃO 2: EXTRAÇÃO DE DADOS ESTRUTURADOS (Séries Tabeladas de Vazões Mensais)**
####Nesta seção, implementamos rotinas para extrair séries temporais de três fontes distintas. O desafio aqui não é apenas ler os números, mas normalizar tabelas de PDFs (muitas vezes fragmentadas ou mal formatadas) em DataFrames limpos, permitindo a análise estatística subsequente na Seção 3.
###**SEÇÃO 2.1**. Extração da Série de Vazões da Outorga
####Captura os dados oficiais vigentes na publicação da ooutorga do empreendimento. O script lê as páginas específicas do PDF de Outorga e persiste os dados na aba OUTORGA do arquivo COMPARA.xlsx.
###**SEÇÃO 2.2**. Extração da lista de aproveitamentos do relatório (PDF) do ONS:
#### Nesta seção é extraída a lista de Aproveitamentos Hidrelétricos do relatório anual deconsolidação deséries de vazões naturais elaboradopelo ONS. A partir dessa lista, são realizados cruzamentos pelo código dos aproveitamentos para adeterminação do nome da série dedados utilizada na planilha de dados do ONS.
####Esta subseção resolve o problema de inconsistência de nomes entre o relatório em PDF do ONS e o identificador usado na planilha de dados em Excel, do ONS.
####Nesta seção foi usada a biblioteca pdfplumber (mais robusta que o PDFTableReader para este caso específico) para listar todos os aproveitamentos.
####Cruzamento (Join): Realiza um merge entre a lista extraída e a planilha mestre do ONS, preenchendo o Identificador_ONS na aba APROVEITAMENTOS. Isso garante que estamos comparando a usina correta, independentemente de variações no nome.
####Conforme observado no código, tabelas densas e com linhas de grade complexas (como as do ONS) costumam "quebrar" em leitores convencionais. O pdfplumber foi adotado aqui por sua capacidade de interpretar estratégias verticais e horizontais de linhas de grade (snap_tolerance), essencial para não perder colunas de dados hídricos.
###**SEÇÃO 2.3**. Extração da série de dados consolidados do ONS
####Implementa uma lógica de decisão em duas camadas:
####Automática: Tenta vincular o nome extraído do processo (Tag [a]) à base do ONS via Regex.
####Manual (Fallback): Caso haja ambiguidade (ex: usinas no mesmo rio), o sistema apresenta uma lista numerada para seleção do usuário.
###**SEÇÃO 2.4**. Extração da Série de Vazões Proposta pelo Empreendedor
####Localiza e extrai a série de vazões que o empreendedor submeteu à análise regulatória. O sistema prioriza o arquivo processado por OCR (processo_OCR.pdf) para garantir a leitura de tabelas digitalizadas como imagem, aplicando centralização e normalização numérica automática nos dados resultantes.
###**SEÇÃO 2.4**. Extração do portal Dados Abertos
####Nesta seção, é realizada conexão ao portal DADOS ABERTOS para a obtenção das séries dedados fluviométricos e de operação de reservatórios tanto da ANA quanto do ONS.

In [ ]:
# Subseção 2.1 Extração da série de vazões da outorga
print("💡 Iniciando Subseção 2.1 Extração da série de vazões da outorga...")

# Extração e persistência de dados de outorga
if __name__ == "__main__":

    if not os.path.exists(ARQUIVO_OUTORGA):
        raise FileNotFoundError(
            f"Arquivo de outorga não encontrado: {ARQUIVO_OUTORGA}"
        )

    # 1. Extração
    df_outorga = extrair_vazoes(
        caminho_pdf=ARQUIVO_OUTORGA,
        paginas=PIPELINE_CONTEXT["extracao_outorga"]["paginas"]
    )

    # 2. Persistência
    sucesso = salvar_aba_excel(
        df=df_outorga,
        caminho_excel=PIPELINE_CONTEXT["query_engine"]["persistencia"],
        nome_aba=PIPELINE_CONTEXT["extracao_outorga"]["nome_aba"]
    )

    # 3. Diagnóstico da persistência
    diagnosticar_excel(
        caminho_excel=PIPELINE_CONTEXT["query_engine"]["persistencia"],
        nome_aba=PIPELINE_CONTEXT["extracao_outorga"]["nome_aba"]
    )
    if not sucesso:
        raise RuntimeError(
            "Falha ao salvar a aba OUTORGA no Excel."
        )

    print("✅ Subseção 2.1 concluída com sucesso.")

In [ ]:
#Subseção 2.2 Extração da lista de aproveitamentos do ONS
print("💡 Iniciando Subseção 2.2 Extração da lista de aproveitamentos do ONS...")
print("IMPORTANTE! A função extrair_vazoes com PDFTableReader não funcionou bem neste caso. Foi necessário desenvolver método específico com pdfplumber")
# Páginas a serem processadas
PAGINAS = PIPELINE_CONTEXT["extracao_PDF_ONS"]["paginas"] # Valores definidos manualmente. Trabalhar para a detecção automática dessas páginas, futuramente...

#Criando as funções específicas para a subseção 2.2...

def extrair_e_consolidar_tabelas(tabela_pdf, paginas):
    """
    Extrai tabelas de um intervalo de páginas de um PDF usando pdfplumber
    e as consolida em um único DataFrame.

    Parâmetros:
    - tabela_pdf (str): caminho do arquivo PDF
    - paginas (str): intervalo no formato 'inicio-fim' (ex: '4-6')

    Retorna:
    - pd.DataFrame ou None
    """

    print(f"📄 Iniciando extração de tabelas do arquivo:\n{tabela_pdf}")
    print(f"📑 Páginas a serem processadas: {paginas}")

    lista_dataframes = []

    try:
        if "-" not in paginas:
            raise ValueError("O parâmetro 'paginas' deve estar no formato 'inicio-fim'.")

        start, end = map(int, paginas.split("-"))

        if start > end:
            raise ValueError("Página inicial maior que a página final.")

        with pdfplumber.open(tabela_pdf) as pdf:
            total_paginas_pdf = len(pdf.pages)

            if start < 1 or end > total_paginas_pdf:
                raise ValueError(
                    f"O PDF possui {total_paginas_pdf} páginas. "
                    f"Intervalo solicitado: {start}-{end}."
                )

            for i in range(start - 1, end):
                page = pdf.pages[i]

                tabelas = page.extract_tables(
                    table_settings={
                        "vertical_strategy": "lines",
                        "horizontal_strategy": "lines",
                        "snap_tolerance": 3,
                    }
                )

                for tabela in tabelas or []:
                    df_tabela = pd.DataFrame(tabela)
                    lista_dataframes.append(df_tabela)

        if not lista_dataframes:
            print("⚠️ Nenhuma tabela encontrada no intervalo informado.")
            return None

        df_final = pd.concat(lista_dataframes, ignore_index=True)

        print(
            f"✅ Consolidação concluída: "
            f"{df_final.shape[0]} linhas × {df_final.shape[1]} colunas."
        )

        return df_final

    except FileNotFoundError:
        print(f"❌ ERRO: Arquivo não encontrado: {tabela_pdf}")
        return None

    except ValueError as ve:
        print(f"❌ ERRO DE PARÂMETRO: {ve}")
        return None

    except Exception as e:
        print(f"❌ ERRO inesperado durante a extração: {e}")
        return None

def limpar_e_salvar_para_excel(df, caminho_excel, nome_aba, verbose=True):
    """
    Limpa, corrige e organiza um DataFrame extraído de PDF
    e salva o resultado em uma aba Excel padronizada.

    Parâmetros:
    - df (pd.DataFrame): dados extraídos do PDF
    - caminho_excel (str): caminho do arquivo Excel
    - nome_aba (str): nome da aba a ser criada/substituída
    - verbose (bool): controla a exibição de mensagens e previews
    """

    if df is None or df.empty:
        if verbose:
            print("⚠️ DataFrame vazio ou inexistente. Nada a processar.")
        return False

    try:
        # 1. Pré-processamento
        df.columns = [str(col) for col in df.columns]
        df_str = df.astype(str).apply(lambda x: x.str.strip(), axis=1)

        # 2. Identificação de cabeçalho
        indice_cabecalho = -1
        cabecalho_desejado = None

        for i in range(min(10, len(df_str))):
            row = df_str.iloc[i].tolist()
            if len(row) > 4 and 'Cód' in row[0] and 'Bacia' in row[2]:
                indice_cabecalho = i
                cabecalho_desejado = row
                break

        if indice_cabecalho != -1:
            header_tuple = tuple(cabecalho_desejado)
            is_header = df_str.apply(lambda r: tuple(r) == header_tuple, axis=1)
            df = df[~is_header].reset_index(drop=True)
            df.columns = cabecalho_desejado
            if verbose:
                print(f"🧹 {is_header.sum()} linhas de cabeçalho removidas.")
        else:
            if verbose:
                print("⚠️ Cabeçalho não identificado. Promovendo primeira linha.")
            df = df_str.iloc[1:]
            df.columns = df_str.iloc[0].tolist()

        if df.empty:
            if verbose:
                print("⚠️ DataFrame vazio após limpeza inicial.")
            return False

        # 3. Normalização de nomes de colunas
        df.columns = [
            'Cód' if col.lower().startswith('cód') else
            'Nome' if col.lower().startswith('nome') else col
            for col in df.columns
        ]

        # 🔒 CHECAGEM DEFENSIVA DAS COLUNAS CRÍTICAS
        colunas_criticas = {'Cód', 'Nome'}
        colunas_ausentes = colunas_criticas - set(df.columns)

        if colunas_ausentes:
            raise KeyError(
                f"Coluna(s) obrigatória(s) ausente(s) após limpeza: {colunas_ausentes}"
            )

        df = df.replace(['None', ''], np.nan)

        # 4. Correção de linhas divididas
        linhas_remover = []

        for i in range(len(df) - 1, 0, -1):
            if pd.isna(df.at[i, 'Cód']) and pd.notna(df.at[i, 'Nome']):
                df.at[i - 1, 'Nome'] = (
                    f"{df.at[i - 1, 'Nome']} {df.at[i, 'Nome']}".strip()
                )
                for col in df.columns[2:]:
                    if pd.isna(df.at[i - 1, col]) and pd.notna(df.at[i, col]):
                        df.at[i - 1, col] = df.at[i, col]
                linhas_remover.append(i)

        df.drop(linhas_remover, inplace=True)
        df.reset_index(drop=True, inplace=True)

        if verbose:
            print(f"🔧 {len(linhas_remover)} linhas divididas corrigidas.")

        # 5. Limpeza final e ordenação
        df['Cód_Num'] = pd.to_numeric(df['Cód'], errors='coerce')
        linhas_antes = len(df)
        df.dropna(subset=['Cód_Num'], inplace=True)

        if verbose:
            print(f"🗑️ {linhas_antes - len(df)} linhas sem código removidas.")

        df['Cód'] = df['Cód_Num'].astype(int)
        df.drop(columns='Cód_Num', inplace=True)

        df.sort_values(by='Cód', inplace=True)
        df.reset_index(drop=True, inplace=True)

        # 6. Coluna derivada
        df['Cód_e_Nome'] = (
            df['Nome'].astype(str) + " (" + df['Cód'].astype(str) + ")"
        )

        cols = df.columns.tolist()
        cols.remove('Cód_e_Nome')
        cols.insert(2, 'Cód_e_Nome')
        df = df[cols]

        # 7. Persistência (PADRÃO ÚNICO)
        salvar_aba_excel(
            df=df,
            caminho_excel=caminho_excel,
            nome_aba=PIPELINE_CONTEXT["extracao_PDF_ONS"]["nome_aba"]
        )

        if verbose:
            print(f"✅ Aba '{nome_aba}' salva com sucesso em:\n{caminho_excel}")
            print(df.head().to_markdown(index=False))

        return True

    except Exception as e:
        print(f"❌ Erro durante limpeza e salvamento: {e}")
        return False

# Extração de dados da planilha do PDF do ONS
if __name__ == "__main__":

    if not os.path.exists(ARQUIVO_PDF_ONS):
        raise FileNotFoundError(
            f"Arquivo PDF não encontrado: {ARQUIVO_PDF_ONS}"
        )

    # 1. Extração
    df_vazoes = extrair_e_consolidar_tabelas(
        tabela_pdf=ARQUIVO_PDF_ONS,
        paginas=PAGINAS
    )

    if df_vazoes is None or df_vazoes.empty:
        raise RuntimeError("Falha na extração das tabelas do PDF.")

    # 2. Limpeza + persistência
    sucesso = limpar_e_salvar_para_excel(
        df=df_vazoes,
        caminho_excel=PIPELINE_CONTEXT["query_engine"]["persistencia"],
        nome_aba=PIPELINE_CONTEXT["extracao_PDF_ONS"]["nome_aba"]
    )

    if not sucesso:
        raise RuntimeError("Falha ao salvar a aba APROVEITAMENTOS.")

    # 3. Diagnóstico da persistência
    diagnosticar_excel(
        caminho_excel=PIPELINE_CONTEXT["query_engine"]["persistencia"],
        nome_aba=PIPELINE_CONTEXT["extracao_PDF_ONS"]["nome_aba"]
    )

print("\n💡 Cruzamento de códigos da tabela extraída do PDF do ONS com a planilha Excel do ONS...")

try:
    # 1. Carregar a planilha ONS (Vazões_Mensais_1931_2024.xls)
    # Título na linha 7 (index 6), dados começam abaixo
    df_ons_raw = pd.read_excel(ARQUIVO_EXCEL_ONS, sheet_name="Tabela", skiprows=6)

    # Filtrar apenas as linhas onde a coluna 'Aprov.' não é nula
    # Isso elimina as células vazias entre os registros (linhas 9, 107, 205...)
    df_ons = df_ons_raw[df_ons_raw['Aprov.'].notna()].copy()

    # 2. Função para extrair apenas o número dentro dos parênteses
    def extrair_id(texto):
        match = re.search(r'\((\d+)\)', str(texto))
        return int(match.group(1)) if match else None

    # Criar uma coluna temporária com o ID numérico para o cruzamento
    df_ons['cod_extraido'] = df_ons['Aprov.'].apply(extrair_id)

    # 3. Carregar a planilha APROVEITAMENTOS (COMPARA.xlsx)
    df_aprov = pd.read_excel(COMPARA, sheet_name="APROVEITAMENTOS")

    # 4. Realizar o "Merge" (Cruzamento)
    # Cruzamos 'Cód' da COMPARA com 'cod_extraido' da ONS
    df_resultado = pd.merge(
        df_aprov,
        df_ons[['cod_extraido', 'Aprov.']],
        left_on='Cód',
        right_on='cod_extraido',
        how='left'
    )

    # A coluna J desejada será a 'Aprov.' que veio do arquivo da ONS
    # Renomeamos para ficar claro e removemos a coluna auxiliar
    df_resultado['Identificador_ONS'] = df_resultado['Aprov.']
    df_resultado = df_resultado.drop(columns=['cod_extraido', 'Aprov.'])

    # 5. Persistência de volta no Excel mantendo as outras abas
    with pd.ExcelWriter(COMPARA, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        df_resultado.to_excel(writer, sheet_name="APROVEITAMENTOS", index=False)

    # 6. Formatação Estética (Wrap Text e Alinhamento)
    wb = load_workbook(COMPARA)
    ws = wb["APROVEITAMENTOS"]

    # Ajustar largura da nova coluna J (Identificador_ONS)
    ws.column_dimensions['J'].width = 35

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical='top', horizontal='left')

    wb.save(COMPARA)

    print(f"✅ Sucesso! {len(df_resultado)} registros processados.")
    print(f"✅ Coluna J ('Identificador_ONS') preenchida na aba APROVEITAMENTOS.")

except Exception as e:
    print(f"❌ Erro ao cruzar dados: {e}")

print("\n✅ Extração de dados da Subseção 2.2 concluída com sucesso.")

In [ ]:
# Subsessão 2.3 – Extração da série de vazões naturais do ONS

resposta_rio = PIPELINE_CONTEXT["extracao_processo"]["respostas"]["[e]"]
rio = resposta_rio[3]
print(f"rio: {rio}")
rio_limpo = str(rio).upper().replace("RIO", "").strip()
print(f"rio_limpo: {rio_limpo}")
df_aprov = pd.read_excel(COMPARA, sheet_name="APROVEITAMENTOS")
for i, col in enumerate(df_aprov.columns):
    amostra = df_aprov.iloc[0, i]
mask_rio = df_aprov.apply(lambda row: row.astype(str).str.contains(rio_limpo, case=False).any(), axis=1).sum()
print(f"\nO termo '{rio_limpo}' aparece em {mask_rio} linhas da planilha no total.")

resposta_usina = PIPELINE_CONTEXT["extracao_processo"]["respostas"]["[a]"]
print(f"resposta_usina: {resposta_usina}")
usina_bruto = str(resposta_usina).upper().strip()
print(f"usina_bruto: {usina_bruto}")
match_uhe = re.search(r'\bUHE\s+([\w\.]+)', usina_bruto)
print(f"match_uhe: {match_uhe}")
usina_limpo = match_uhe.group(1) if match_uhe else usina_bruto.split()[0]
print(f"usina_limpo: {usina_limpo}")
mask_usina = (df_aprov.iloc[:, 1].str.contains(usina_limpo, na=False, case=False)) & \
             (df_aprov.iloc[:, 9].str.contains(usina_limpo, na=False, case=False))
df_perfeito = df_aprov[mask_rio & mask_usina]

# Se encontrou EXATAMENTE uma linha, temos o identificador automático

if len(df_perfeito) == 1:
    identificador_ons = str(df_perfeito.iloc[0, 9])
    nome_UHE = str(df_perfeito.iloc[0, 1])
    print(f"nome_uhe: {nome_uhe}")
    PIPELINE_CONTEXT["extracao_ONS"]["nome_UHE"] = nome_UHE
    PIPELINE_CONTEXT["extracao_ONS"]["identificador_ONS"] = identificador_ons
    print(f"✅ Match Automático! Usina: {df_perfeito.iloc[0, 1]} | ID ONS: {identificador_ons}")

# --- PASSO B: SELEÇÃO MANUAL (Se falhar ou houver ambiguidade) ---
else:
    print(f"⚠️ Não foi possível determinar automaticamente (Encontrados: {len(df_perfeito)}).")
    print(f"Filtrando todas as usinas do rio '{rio_limpo}' para seleção manual...")

    # Lista todas as usinas que pertencem ao rio_limpo (Coluna 4)
    df_lista_rio = df_aprov[df_aprov.iloc[:, 4].str.contains(rio_limpo, na=False, case=False)].copy()

    if df_lista_rio.empty:
        print(f"❌ Erro crítico: Rio '{rio_limpo}' não localizado na base ONS.")
    else:
        print("\n--- SELEÇÃO DE USINA ---")
        for i, nome in enumerate(df_lista_rio.iloc[:, 1]):
            id_lista = df_lista_rio.iloc[i, 9]
            print(f"[{i}] {nome} (ID: {id_lista})")

        idx = int(input("\n👉 Digite o número da usina correta: "))

        # Salva o resultado da escolha manual
        identificador_ons = str(df_lista_rio.iloc[idx, 9])
        nome_UHE = str(df_lista_rio.iloc[idx, 1])
        PIPELINE_CONTEXT["extracao_ONS"]["nome_UHE"] = nome_UHE
        PIPELINE_CONTEXT["extracao_ONS"]["identificador_ONS"] = identificador_ons
        print(f"✅ Definido manualmente: ID ONS {identificador_ons}")

arquivo_excel_ons = PIPELINE_CONTEXT["execucao"]["paths"]["excel_ons"]
df_origem = pd.read_excel(arquivo_excel_ons, sheet_name=0, header=None)
id_selecionado = identificador_ons
linhas = df_origem[df_origem[0].astype(str) == str(id_selecionado)].index
if linhas.empty:
    raise ValueError(f"Identificador '{id_selecionado}' não encontrado na coluna A.")
else:
    linha_inicio = linhas[0]
    # Cálculo do tamanho da série
    ano_inicial = PIPELINE_CONTEXT["extracao_ONS"]["ano_inicial"]
    ano_final = PIPELINE_CONTEXT["extracao_ONS"]["ano_final"]
    n_anos = ano_final - ano_inicial + 1
    n_linhas = n_anos + 3 # Anos + Mín, Méd, Máx
    linha_fim = linha_inicio + n_linhas
    print(f"📐 Anos: {n_anos} | Linhas esperadas: {n_linhas}")

    # Seleção dos dados (colunas B–Q -> índices 1 a 17)
    df = df_origem.iloc[linha_inicio:linha_fim, 1:17].copy()

    # Nomeação das colunas
    colunas = ['Ano'] + [f'Mês_{i}' for i in range(1, 13)] + ['Vazão_Anual', 'Col_P', 'Col_Q']
    df.columns = colunas[:len(df.columns)]

    # Criação da coluna de Tipo para facilitar filtros futuros
    df['Tipo'] = (['Ano'] * n_anos + ['Mínimo_Série', 'Média_Série', 'Máximo_Série'])[:len(df)]

    # Reorganização das colunas
    cols = ['Ano', 'Tipo'] + [c for c in df.columns if c not in ('Ano', 'Tipo')]
    df = df[cols]

    print(f"✅ Série extraída com sucesso: {df.shape[0]} linhas.")

salvar_aba_excel(
    df=df,
    caminho_excel=PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"],
    nome_aba=PIPELINE_CONTEXT["extracao_ONS"]["nome_aba"]
)

diagnosticar_excel(
    caminho_excel=PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"],
    nome_aba=PIPELINE_CONTEXT["extracao_ONS"]["nome_aba"]
)

caminho_excel = PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"]
wb = load_workbook(caminho_excel)
nome_aba_ons = PIPELINE_CONTEXT["extracao_ONS"]["nome_aba"]

if nome_aba_ons in wb.sheetnames:
    ws_ons = wb[nome_aba_ons]

    # 2 - Deletar a coluna B
    # O segundo argumento (idx) é o índice da coluna (1 para A, 2 para B...)
    ws_ons.delete_cols(2)

    # 3 - Salvar
    wb.save(caminho_excel)
    print(f"✅ Coluna B da aba '{nome_aba_ons}' eliminada com sucesso.")
else:
    print(f"⚠️ Aba '{nome_aba_ons}' não encontrada no arquivo.")

In [ ]:
# Subsequção 2.4 Extração da série de vazões proposta pelo empreendedor
print("💡 Iniciando Subseção 2.4 Extração da série de vazões proposta pelo empreendedor...")

# 1. Definição do arquivo do processo (Prioridade OCR)
caminho_original = PIPELINE_CONTEXT["execucao"]["paths"]["pdf_processo"]
caminho_ocr = os.path.join(RAIZ, 'processo_OCR.pdf')
CAMINHO_FINAL = caminho_ocr if os.path.exists(caminho_ocr) else caminho_original

print(f"📂 Arquivo de origem: {os.path.basename(CAMINHO_FINAL)}")

# 2. Tentativa de Localização Automática
paginas_vazoes = localizar_paginas_vazoes_dinamico(CAMINHO_FINAL)

# 3. Lógica de Fallback
if not paginas_vazoes:
    print("\n" + "!"*60)
    print("⚠️  AVISO: Não conseguimos localizar a tabela automaticamente.")
    print("Isso pode ocorrer se a tabela for uma imagem não processada ou tiver formato atípico.")

    # Input manual como fallback
    paginas_vazoes = input("👉 Por favor, digite as páginas da tabela (ex: 32,33) ou 'pular': ").strip()
    print("!"*60 + "\n")

# 4. Execução da Extração (apenas se não decidiu pular)
if paginas_vazoes.lower() != 'pular' and paginas_vazoes != "":
    print(f"\n💡 Extraindo dados das páginas: {paginas_vazoes}...")

    df_empreendedor = extrair_vazoes(
        caminho_pdf=CAMINHO_FINAL,
        paginas=paginas_vazoes
    )

    if df_empreendedor is not None:
        # Salva no Excel
        sucesso = salvar_aba_excel(
            df=df_empreendedor,
            caminho_excel=PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"],
            nome_aba=PIPELINE_CONTEXT["extracao_empreendedor"]["nome_aba"]
        )
        wb = load_workbook(PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"])
        ws = wb[PIPELINE_CONTEXT["extracao_empreendedor"]["nome_aba"]]

        for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=13):
            for cell in row:
                cell.alignment = Alignment(horizontal='center', vertical='center')

        wb.save(PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"])
        print("🎯 Tabela centralizada e valores normalizados!")
        print("\n✅ Extração da série de vazões proposta pelo empreendedor concluída.")
    else:
        print("❌ Falha na extração dos dados das páginas fornecidas.")
        todas_as_linhas = []
        config = {
            "vertical_strategy": "text",
            "horizontal_strategy": "text",
            "snap_y_tolerance": 5,
        }
        with pdfplumber.open(CAMINHO_FINAL) as pdf:
            for i in range(17, 21):
                pagina = pdf.pages[i]
                tabela = pagina.extract_table(table_settings=config)
                if tabela:
                    linhas_limpas = [l for l in tabela if any(str(c).strip() for c in l if c)]
                    todas_as_linhas.extend(linhas_limpas)
                    print(f"✅ Página {i}: {len(linhas_limpas)} linhas capturadas.")

        if not todas_as_linhas:
            print("❌ Nada foi capturado.")


        # 1. Cria o DataFrame sem nomes de colunas ainda
        df = pd.DataFrame(todas_as_linhas)

        # 2. Limpeza de colunas/linhas fantasmas
        df = df.dropna(axis=1, how='all').dropna(axis=0, how='all')

        # 3. Pega todas as colunas que vieram no PDF
        num_colunas_reais = df.shape[1]
        print(f"📊 O PDF entregou {num_colunas_reais} colunas.")

        # 4. Remove linhas que repetem o cabeçalho
        df = df[df[0].astype(str).str.contains("Ano|ano") == False]

        # 5. Lista completa de nomes possíveis
        nomes_possiveis = ['Ano', 'Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez', 'Media_Anual', 'Extra_1', 'Extra_2']

        # Ajusta os nomes de acordo com a quantidade de colunas que o PDF realmente tem
        df.columns = nomes_possiveis[:num_colunas_reais]

        # 6. Limpa espaços e salva
        df = df.map(lambda x: str(x).strip() if x else x)

        # 7. Salva no Excel
        salvar_aba_excel(
            df=df,
            caminho_excel=PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"],
            nome_aba=PIPELINE_CONTEXT["extracao_empreendedor"]["nome_aba"]
        )
        wb = load_workbook(PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"])
        ws = wb[PIPELINE_CONTEXT["extracao_empreendedor"]["nome_aba"]]

        for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=14):
            for cell in row:
                cell.alignment = Alignment(horizontal='center', vertical='center')

        wb.save(PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"])
        print("🎯 Tabela centralizada e valores normalizados!")
        print("\n✅ Extração da série de vazões proposta pelo empreendedor concluída.")

else:
    print("⏭️ Subseção 2.4 ignorada pelo usuário ou por falta de dados.")

In [ ]:
# Em desenvolvimento

#Extração de dados de estações fluviométricas do portal DADOS ABERTOS

!pip install hydrobr

import requests
import pandas as pd
import xml.etree.ElementTree as ET

# 1. Configuração dos parâmetros
url = "http://telemetriaws1.ana.gov.br/ServiceANA.asmx/HidroSerieHistorica"
params = {
    "codEstacao": "66070004",  # Exemplo: Estação no Rio Paraguai
    "dataInicio": "01/01/2023",
    "dataFim": "01/01/2024",
    "tipoDados": "3",          # 3 para Vazão, 2 para Nível, 1 para Chuva
    "nivelConsistencia": "1"   # 1 para Bruto, 2 para Consistido
}

# 2. Fazendo a requisição (HTTP GET)
response = requests.get(url, params=params)

if response.status_code == 200:
    print("Sucesso! Dados recebidos.")
    # O conteúdo vem em XML bruto
    xml_data = response.content
    # No XML, cada linha é um mês e as colunas são numeradas de 1 a 31 para cada dia.
else:
    print(f"Erro na requisição: {response.status_code}")

# 3. Parsear o XML para extrair a tabela
root = ET.fromstring(xml_data)

# Criar uma lista de dicionários com os dados das séries
dados_lista = []
for serie in root.findall(".//SerieHistorica"):
    # Extrair campos básicos (exemplo: data e vazões do dia)
    data = serie.find("DataHora").text

    # A ANA envia 31 colunas (Vazao01, Vazao02...) para cada dia do mês
    # Aqui pegamos a média ou o valor principal dependendo da sua necessidade
    vazao_media = serie.find("Vazao01").text # Exemplo simplificado

    dados_lista.append({
        "Data": data,
        "Vazao": vazao_media
    })

# 4. Criar o DataFrame
df = pd.DataFrame(dados_lista)
df['Data'] = pd.to_datetime(df['Data'])
print(df.head())

# testar o método get_data do hydrobr
hydrobr.get_data.ANA.flow_data(list_code=['66070004'])

###**Seção 3: PREPARAÇÃO PARA ANÁLISE DA PROPOSIÇÃO (Auditoria Automatizada)**
####Nesta fase, o pipeline injeta fórmulas lógicas diretamente no Excel para realizar uma "prova real" entre as planilhas. Em vez de uma conferência manual exaustiva, o analista passa a focar apenas nas células sinalizadas em vermelho, otimizando o tempo de resposta da Nota Técnica.
###**SEÇÃO 3.1**. Avaliação 1: Diferença de Somas Totais (Vizinhança de Erro)
####Esta função injeta na Coluna R uma fórmula que subtrai a soma anual do empreendedor da soma anual do ONS.
####Critério de Aceitação: Diferenças entre -0.01 e 0.01 são marcadas em Amarelo (indicando arredondamentos desprezíveis).
####Alerta Crítico: Qualquer valor fora desse intervalo é marcado em Vermelho, sinalizando que o volume total anual de água diverge da base oficial.
###**SEÇÃO 3.2**. Avaliação 2: Soma de Resíduos Absolutos (Precisão Mensal)
####Enquanto a Avaliação 1 olha o "todo", a Avaliação 2 (Coluna S) é implacável com os detalhes mensais.
####Lógica: Ela calcula a soma das diferenças absolutas (módulo) de cada mês, tratando erros de texto com IFERROR e truncando decimais com INT para focar na parte inteira da vazão.
####Resultado: Se houver um único mês com valor diferente entre as tabelas, a Coluna S acusará um valor maior que zero e ficará Vermelha. É o teste definitivo de "De acordo" ou "Não de acordo" para a série proposta.
###**SEÇÃO 3.3**. Estilização e Persistência Visual
####O código utiliza a biblioteca openpyxl para garantir que o Excel gerado não seja apenas funcional, mas legível.
####Negrito e Alinhamento: Os títulos das avaliações são destacados e as fórmulas centralizadas.
####Sincronização Dinâmica: As fórmulas utilizam o nome da aba APROVEITAMENTOS capturado no contexto, garantindo que o cruzamento funcione mesmo se o nome da usina mudar.

In [ ]:
# Seção 3 Avaliações de dados das planilhas para análise da proposição

def executar_avaliacao_1_somas(caminho_excel):
    print("💡 Executando Avaliação 1: Diferença de Somas Totais (Coluna R)...")
    wb = load_workbook(caminho_excel)
    ws_emp = wb[PIPELINE_CONTEXT["extracao_empreendedor"]["nome_aba"]]
    nome_aba_ons = PIPELINE_CONTEXT["extracao_ONS"]["nome_aba"]

    # Estilos
    fill_amarelo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
    fill_vermelho = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")
    alinhamento = Alignment(horizontal='center', vertical='center')
    fonte_negrito = Font(bold=True)

    col_av1 = 18 # Coluna R

    # 1. Inserir Título
    titulo = ws_emp.cell(row=1, column=col_av1)
    titulo.value = "AVALIAÇÃO 1"
    titulo.font = fonte_negrito
    titulo.alignment = alinhamento

    # 2. Inserir Fórmulas
    for row in range(2, ws_emp.max_row + 1):
        formula = f"=(SUM(A{row}:M{row})) - (SUM('{nome_aba_ons}'!A{row}:M{row}))"
        celula = ws_emp.cell(row=row, column=col_av1)
        celula.value = formula
        celula.alignment = alinhamento

    # 3. Formatação Condicional (Limpa antes de aplicar)
    ws_emp.conditional_formatting._rules = {} # Limpa regras da aba para evitar sobreposição
    range_av1 = f"R2:R{ws_emp.max_row}"

    # Regra: Amarelo se for 0 (dentro da margem de erro)
    ws_emp.conditional_formatting.add(range_av1,
        FormulaRule(formula=['AND(R2<=0.01; R2>=-0.01)'], stopIfTrue=True, fill=fill_amarelo))

    # Regra: Vermelho se for diferente de 0
    ws_emp.conditional_formatting.add(range_av1,
        FormulaRule(formula=['OR(R2>0.01; R2<-0.01)'], stopIfTrue=True, fill=fill_vermelho))

    wb.save(caminho_excel)
    print("✅ Avaliação 1 concluída com títulos e cores.")

def executar_avaliacao_2_residuos(caminho_excel):
    print("💡 Executando Avaliação 2: Soma de Resíduos Absolutos (Coluna S)...")
    wb = load_workbook(caminho_excel)
    ws_emp = wb[PIPELINE_CONTEXT["extracao_empreendedor"]["nome_aba"]]
    nome_aba_ons = PIPELINE_CONTEXT["extracao_ONS"]["nome_aba"]

    # Estilos
    fill_amarelo = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
    fill_vermelho = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")
    alinhamento = Alignment(horizontal='center', vertical='center')
    fonte_negrito = Font(bold=True)

    col_av2 = 19 # Coluna S

    # 1. Inserir Título
    titulo = ws_emp.cell(row=1, column=col_av2)
    titulo.value = "AVALIAÇÃO 2"
    titulo.font = fonte_negrito
    titulo.alignment = alinhamento

    letras_colunas = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M']

    # 2. Inserir Fórmulas
    for row in range(2, ws_emp.max_row + 1):
        partes = []
        for letra in letras_colunas:
            p = f"ABS(IFERROR(INT({letra}{row});0)-IFERROR(INT('{nome_aba_ons}'!{letra}{row});0))"
            partes.append(p)

        formula_final = "=" + "+".join(partes)
        celula = ws_emp.cell(row=row, column=col_av2)
        celula.value = formula_final
        celula.alignment = alinhamento

    # 3. Formatação Condicional
    range_av2 = f"S2:S{ws_emp.max_row}"

    # Regra: Amarelo se for 0
    ws_emp.conditional_formatting.add(range_av2,
        FormulaRule(formula=['S2=0'], stopIfTrue=True, fill=fill_amarelo))

    # Regra: Vermelho se for diferente de 0
    ws_emp.conditional_formatting.add(range_av2,
        FormulaRule(formula=['S2<>0'], stopIfTrue=True, fill=fill_vermelho))

    wb.save(caminho_excel)
    print("✅ Avaliação 2 concluída com títulos e cores.")

executar_avaliacao_1_somas(PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"])
executar_avaliacao_2_residuos(PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"])

###**SEÇÃO 4: PREPARAÇÃO DA MINUTA E FINALIZAÇÃO DO CASO**
####Nesta etapa, consolidamos todo o conhecimento extraído nas seções anteriores. O pipeline realiza a fusão de dados estruturados e não estruturados em um documento Microsoft Word (.docx) e executa o "limpador de trilhas", organizando os arquivos em pastas versionadas para auditoria futura.
###**SEÇÃO 4.1**. Preenchimento Automático da Nota Técnica
####Esta função utiliza o python-docx para realizar uma varredura inteligente no template padrão da agência.
####Substituição de Tags: O motor de busca percorre cada parágrafo e tabela, trocando etiquetas como [a], [b], [i] pelos valores reais extraídos pelo RAG e pela Heurística.
####Padronização: Garante que nomes de usinas e números de processos sejam escritos de forma idêntica em todo o documento, eliminando erros de digitação comuns em preenchimentos manuais.
###**SEÇÃO 4.2**. Geração de Auditoria e Snapshot (Ground Truth)
####Para fins de conformidade e transparência, o sistema gera dois registros do momento da execução:
####JSON de Auditoria: Um "raio-x" técnico do PIPELINE_CONTEXT contendo todos os parâmetros de IA e caminhos de sistema utilizados.
####Resumo para Humanos (.txt): Um relatório simplificado com os dados mestre (UHE, Rio, ID ONS) e as respostas extraídas, facilitando uma conferência rápida sem abrir o Excel.

In [ ]:
def preencher_word_com_respostas():
    # 1. Recupera os dados
    caminhos = PIPELINE_CONTEXT["execucao"]["paths"]
    respostas = PIPELINE_CONTEXT["extracao_processo"]["respostas"]
    nome_usina_bruto = PIPELINE_CONTEXT["extracao_ONS"]["nome_UHE"]

    # Se o valor for None, define um fallback
    if not nome_usina_bruto: nome_usina_bruto = "USINA_DESCONHECIDA"

    # 3. Limpeza do nome do arquivo (Permanece igual, mas agora usando a string pura)
    nome_usina_limpo = re.sub(r'[^\w\s-]', '', str(nome_usina_bruto))
    nome_usina_limpo = nome_usina_limpo.strip().replace(" ", "_")

    nome_saida = f"NOTA_TECNICA_UHE_{nome_usina_limpo}.docx"
    DOCX_SAIDA_LOCAL = os.path.join(RAIZ, nome_saida)

    # 4. Carregar o Template Word
    caminho_template = caminhos.get("nota_template")
    if not caminho_template or not os.path.exists(caminho_template):
        print(f"❌ Erro: Template não encontrado em {caminho_template}")
        return

    doc = docx.Document(caminho_template)
    print(f"🏗️ Processando Usina: {nome_usina_bruto}")

    # 5. Substituição de Tags (Ajustado para buscar o índice [3])
    def realizar_substituicao(container):
        for paragrafo in container:
            for etiqueta, meta in respostas.items():
                # Extrai apenas o texto da resposta (índice 3)
                valor_final = meta[3] if isinstance(meta, list) else meta

                # Se o valor for None, tratamos como string vazia
                if valor_final is None: valor_final = ""

                if etiqueta in paragrafo.text:
                    # Faz a troca do texto da tag pelo valor da resposta puro
                    paragrafo.text = paragrafo.text.replace(etiqueta, str(valor_final))

    realizar_substituicao(doc.paragraphs)
    for tabela in doc.tables:
        for linha in tabela.rows:
            for celula in linha.cells:
                realizar_substituicao(celula.paragraphs)

    # 6. Salvar o arquivo
    doc.save(DOCX_SAIDA_LOCAL)
    PIPELINE_CONTEXT["execucao"]["paths"]["nota_final"] = DOCX_SAIDA_LOCAL

    print(f"💾 Documento salvo em: {nome_saida}")


# Executa a função
preencher_word_com_respostas()

In [ ]:
# Organização dos arquivos
def gerar_pdf_contexto():
    # 1. Preparar o conteúdo
    conteudo_texto = pprint.pformat(PIPELINE_CONTEXT, indent=2, width=75)

    # 2. Definir caminhos
    p = PIPELINE_CONTEXT["execucao"]["paths"]
    diretorio_alvo = p.get("diretorio_uhe_trabalho")
    nome_relatorio = "CONTEXTO_PIPELINE_" + PIPELINE_CONTEXT["subdirs"]["UHE"]["caso"] + ".pdf"
    caminho_final = os.path.join(diretorio_alvo, nome_relatorio)

    # 3. Criar o PDF com PyMuPDF
    doc = fitz.open()  # Cria um novo PDF vazio

    # Configurações de fonte e margem
    font_size = 10
    margin = 50
    # Usamos uma fonte monoespaçada para manter a indentação do pprint
    font_name = "cour" # Courier

    # Definimos o retângulo onde o texto PODE aparecer
    # fitz.Rect(margem_esquerda, margem_superior, margem_direita, margem_inferior)
    # A4 padrão tem largura de 595 e altura de 842
    rect = fitz.Rect(margin, margin, 595 - margin, 842 - margin)

    # Dividir o texto em linhas para lidar com múltiplas páginas
    linhas = conteudo_texto.split('\n')

    while linhas:
        page = doc.new_page()  # Cria uma página A4 padrão
        # Em vez de gerenciar linhas manualmente, vamos usar o textbox
        # O PyMuPDF vai tentar encaixar o máximo de texto possível no retângulo

        # Unimos as linhas restantes
        texto_restante = "\n".join(linhas)

        # O método insert_textbox retorna a quantidade de texto que NÃO coube (overflow)
        # Mas para facilitar o controle de páginas, usaremos o 'text_box' com batch:

        linhas_por_pagina = 65 # Reduzimos um pouco para garantir segurança
        batch = linhas[:linhas_por_pagina]
        linhas = linhas[linhas_por_pagina:]
        texto_pagina = "\n".join(batch)

        page.insert_textbox(
            rect,
            texto_pagina,
            fontsize=font_size,
            fontname=font_name,
            align=0 # 0 = Alinhado à esquerda
        )

        # Rodapé
        page.insert_text(
            (margin, 842 - 20),
            f"Contexto de processamento - {datetime.now().strftime('%d/%m/%Y %H:%M')} - Página {page.number + 1}",
            fontsize=6
        )
    # 4. Salvar
    doc.save(caminho_final)
    doc.close()
    print(f"✅ PDF gerado com PyMuPDF: {caminho_final}")

def persistir_e_limpar_ambiente_caso():
    print("\n📦 Iniciando persistência final e organização do caso...")

    # --- 1. VALIDAÇÃO E DEFINIÇÃO DE NOMES ---
    respostas = PIPELINE_CONTEXT["extracao_processo"]["respostas"]

    # Usamos o nome oficial do ONS (mais limpo) para os arquivos
    nome_ons = PIPELINE_CONTEXT["extracao_ONS"]["nome_UHE"]
    nome_slug = re.sub(r'[^\w\s-]', '', str(nome_ons)).strip().replace(" ", "_").upper()

    carimbo_tempo = datetime.now().strftime('%Y%m%d')

    # Definição dos nomes finais
    nome_diretorio = f"UHE_{nome_slug}_{carimbo_tempo}"
    PIPELINE_CONTEXT["subdirs"]["UHE"]["caso"] = nome_diretorio
    novo_nome_excel = f"COMPARA_{nome_slug}.xlsx"

    # --- 2. CRIAÇÃO DO DIRETÓRIO ---
    raiz = PIPELINE_CONTEXT["execucao"]["paths"]["raiz_projeto"]
    caminho_uhe_pasta = os.path.join(raiz, "UHE", nome_diretorio)

    try:
        os.makedirs(caminho_uhe_pasta, exist_ok=True)
        # Atualiza o contexto para a função de auditoria saber onde salvar
        PIPELINE_CONTEXT["execucao"]["paths"]["diretorio_uhe_trabalho"] = caminho_uhe_pasta

        # --- 3. RENOMEAR EXCEL (Ainda na raiz para depois mover) ---
        caminho_excel_atual = PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"]
        caminho_excel_novo = os.path.join(raiz, novo_nome_excel)

        if os.path.exists(caminho_excel_atual):
            os.rename(caminho_excel_atual, caminho_excel_novo)
            # Atualiza o path no contexto para o novo nome
            PIPELINE_CONTEXT["execucao"]["paths"]["excel_saida"] = caminho_excel_novo
            print(f"📊 Excel preparado: {novo_nome_excel}")

        # --- 4. Captura do contexto ---
        gerar_pdf_contexto()


        # --- 5. TRANSFERÊNCIA FINAL DE ARQUIVOS ---
        p = PIPELINE_CONTEXT["execucao"]["paths"]

        # Dicionário: {caminho_origem: nome_destino_dentro_da_pasta}
        arquivos_para_mover = {
            p["excel_saida"]: novo_nome_excel,
            p["pdf_processo"]: os.path.basename(p["pdf_processo"]) if p.get("pdf_processo") else None,
            p["pdf_outorga"]: os.path.basename(p["pdf_outorga"]) if p.get("pdf_outorga") else None,
            p["nota_final"]: os.path.basename(p["nota_final"]) if p.get("nota_final") else None,
            p["persistencia_drive"]: os.path.basename(p["persistencia_drive"]) if p.get("persistencia_drive") else None,
            p["compactado_drive"]: os.path.basename(p["compactado_drive"]) if p.get("compactado_drive") else None,
            p["pdf_processo_ocr"]: os.path.basename(p["pdf_processo_ocr"]) if p.get("pdf_processo_ocr") else None,
            p["pdf_outorga_ocr"]: os.path.basename(p["pdf_outorga_ocr"]) if p.get("pdf_outorga_ocr") else None,
            os.path.join(p["raiz_projeto"], "CONTEXTO_PIPELINE.pdf"): "CONTEXTO_PIPELINE.pdf"
        }

        print("🚚 Movendo arquivos para a pasta final...")
        for origem, nome_destino in arquivos_para_mover.items():
            # 1. Ignora entradas vazias ou None
            if not origem or not nome_destino:
                continue

            if os.path.exists(origem):
                destino_final = os.path.join(caminho_uhe_pasta, nome_destino)

                # Evita erro de tentar copiar o arquivo sobre ele mesmo
                if os.path.abspath(origem) == os.path.abspath(destino_final):
                    print(f"ℹ️ Arquivo já está no destino: {nome_destino}")
                    continue

                try:
                    # 2. Copia com preservação de metadados
                    shutil.copy2(origem, destino_final)

                    # 3. Validação: Só apaga o original se o destino existir e tiver tamanho > 0
                    if os.path.exists(destino_final) and os.path.getsize(destino_final) > 0:
                        os.remove(origem)
                        print(f"✅ Sucesso: {nome_destino}")
                    else:
                        print(f"❌ Falha na integridade: {nome_destino} não foi copiado corretamente.")

                except Exception as e:
                    print(f"⚠️ Erro ao processar {nome_destino}: {e}")
            else:
                print(f"🔍 Ignorado (não encontrado na origem): {os.path.basename(origem)}")

        origem = p["diretorio_faiss_drive"]
        destino = PIPELINE_CONTEXT["subdirs"]["UHE"]["caso"]
        if os.path.exists(origem):
            shutil.move(origem, destino)
            print(f"✅ Sucesso: {os.path.basename(origem)} movido")

    except Exception as e:
        print(f"❌ Erro crítico na persistência final: {e}")
        return False

persistir_e_limpar_ambiente_caso()

In [ ]:
#4.3 Resetar contexto para um novo caso, nova rodada
def resetar_contexto_para_novo_caso():
    """
    Limpa as chaves temporárias do contexto, mantendo as configurações
    fixas de paths e modelos para a próxima rodada.
    """
    # Limpa as respostas extraídas
    PIPELINE_CONTEXT["extracao_processo"]["respostas"] = {}

    # Reseta dados da ONS
    PIPELINE_CONTEXT["extracao_ONS"]["nome_uhe"] = None
    PIPELINE_CONTEXT["extracao_ONS"]["identificador_ONS"] = None

    # Reseta subdiretórios
    PIPELINE_CONTEXT["subdirs"]["UHE"]["caso"] = None

    print("🧹 Contexto resetado. Pronto para o próximo PDF!")

resetar_contexto_para_novo_caso()